#  DRM + Wise-IoU + DWA vs. Epoch-Matched Control


- **Control**: unchanged architecture and loss (continues the paper's exact recipe) --
  isolates "does training longer alone help?"
- **Proposed**: DRM (Detail Recovery Module at P3) + Wise-IoU dynamic focusing + Dynamic
  Weight Averaging (DWA) for the detection/dehaze loss balance -- isolates "does the
  technique help beyond that?"

Both are then evaluated on **VOC-FOG-test** and **RTTS** (FDD intentionally out of scope),
alongside the unmodified baseline checkpoint for reference, using the same subprocess-isolated
`eval_one.py` for every checkpoint so each evaluation always builds the exact architecture
(`USE_DRM`) that checkpoint was trained with -- see `eval_one.py`'s docstring for why this
matters (Python caches module imports; DRM changes `nets/model.py`'s architecture at import
time based on `config.USE_DRM`).

### Before running
1. In the right sidebar, click **+ Add Data** and attach:
   - `mdhabibourrahman/voc-fog`
   - `tuncnguyn/rtts-dataset`
   - `mdhabibourrahman/rdfnet-baseline`
2. **Settings > Accelerator > GPU T4 x2** -- required exactly as-is: `nets/backbone.py`
   hardcodes an `x.split((8, 8), dim=0)` that assumes a 2-GPU DDP split of a
   batch-16-hazy + batch-16-clean pair into 8+8 per GPU. Do not change batch size or GPU
   count without also fixing that hardcoded split.
3. Run all cells top to bottom. **Read the "Set the fine-tune epoch budget" cell before
   launching training** -- the default (`FINETUNE_EPOCHS = 60`) is a placeholder pending your
   own timing calibration; the cell explains how to adjust it after watching the first
   epoch's progress bar.
4. Each training run can take hours; if a Kaggle session times out mid-run, checkpoints are
   saved every `save_period` epochs under `logs_control/` / `logs_proposed/` -- re-attach this
   notebook's `/kaggle/working` output as an input dataset in a new session to resume manually
   (not automated here).

In [ ]:
%cd /kaggle/working
!rm -rf adverse_weather_ibject_detection
!git clone --depth 1 https://github.com/habibour/adverse_weather_ibject_detection.git
%cd /kaggle/working/adverse_weather_ibject_detection/RDFNet

## Apply this thesis's modifications (self-contained, no git push required)

Writes the 6 changed/new files (DRM, Wise-IoU, DWA, config toggles, mAP-based checkpoint
selection, `eval_one.py`) directly on top of the freshly cloned base repo. This does not
depend on `github.com/habibour/adverse_weather_ibject_detection` having been updated --
everything needed is embedded in this notebook.

In [ ]:
import base64, os

_FILES_B64 = {
    'nets/Common.py': (
        'aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRob3AgaW1wb3J0IHByb2ZpbGUgCgpjbGFzcyBTaUxVKG5uLk1vZHVsZSk6CiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZm9yd2FyZCh4KToKICAgICAgICByZXR1cm4geCAqIHRvcmNoLnNp'
        'Z21vaWQoeCkKCmRlZiBhdXRvcGFkKGssIHA9Tm9uZSk6CiAgICBpZiBwIGlzIE5vbmU6CiAgICAgICAgcCA9IGsgLy8gMiBpZiBpc2luc3RhbmNlKGssIGludCkgZWxzZSBbeCAvLyAyIGZvciB4IGluIGtdIAogICAgcmV0dXJuIHAKICAgIApjbGFzcyBDb252KG5u'
        'Lk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYzEsIGMyLCBrPTEsIHM9MSwgcD1Ob25lLCBnPTEsIGFjdD1ubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpKTogICMgY2hfaW4sIGNoX291dCwga2VybmVsLCBzdHJpZGUsIHBhZGRpbmcsIGdyb3Vw'
        'cwogICAgICAgIHN1cGVyKENvbnYsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYgICA9IG5uLkNvbnYyZChjMSwgYzIsIGssIHMsIGF1dG9wYWQoaywgcCksIGdyb3Vwcz1nLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4gICAgID0gbm4uQmF0'
        'Y2hOb3JtMmQoYzIsIGVwcz0wLjAwMSwgbW9tZW50dW09MC4wMykKICAgICAgICBzZWxmLmFjdCAgICA9IG5uLkxlYWt5UmVMVSgwLjEsIGlucGxhY2U9VHJ1ZSkgaWYgYWN0IGlzIFRydWUgZWxzZSAoYWN0IGlmIGlzaW5zdGFuY2UoYWN0LCBubi5Nb2R1bGUpIGVs'
        'c2Ugbm4uSWRlbnRpdHkoKSkKCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2VsZi5jb252KHgpKSkKCiAgICBkZWYgZnVzZWZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNl'
        'bGYuY29udih4KSkKICAgIApjbGFzcyBCYXNpY0NvbnYobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGluX3BsYW5lcywKICAgICAgICBvdXRfcGxhbmVzLAogICAgICAgIGtlcm5lbF9zaXplLAogICAgICAgIHN0cmlk'
        'ZT0xLAogICAgICAgIHBhZGRpbmc9MCwKICAgICAgICBkaWxhdGlvbj0xLAogICAgICAgIGdyb3Vwcz0xLAogICAgICAgIHJlbHU9VHJ1ZSwKICAgICAgICBibj1UcnVlLAogICAgICAgIGJpYXM9RmFsc2UsCiAgICApOgogICAgICAgIHN1cGVyKEJhc2ljQ29udiwg'
        'c2VsZikuX19pbml0X18oKQogICAgICAgIHNlbGYub3V0X2NoYW5uZWxzID0gb3V0X3BsYW5lcwogICAgICAgIHNlbGYuY29udiA9IG5uLkNvbnYyZCgKICAgICAgICAgICAgaW5fcGxhbmVzLAogICAgICAgICAgICBvdXRfcGxhbmVzLAogICAgICAgICAgICBrZXJu'
        'ZWxfc2l6ZT1rZXJuZWxfc2l6ZSwKICAgICAgICAgICAgc3RyaWRlPXN0cmlkZSwKICAgICAgICAgICAgcGFkZGluZz1wYWRkaW5nLAogICAgICAgICAgICBkaWxhdGlvbj1kaWxhdGlvbiwKICAgICAgICAgICAgZ3JvdXBzPWdyb3VwcywKICAgICAgICAgICAgYmlh'
        'cz1iaWFzLAogICAgICAgICkKICAgICAgICBzZWxmLmJuID0gKAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChvdXRfcGxhbmVzLCBlcHM9MWUtNSwgbW9tZW50dW09MC4wMSwgYWZmaW5lPVRydWUpCiAgICAgICAgICAgIGlmIGJuCiAgICAgICAgICAgIGVsc2Ug'
        'Tm9uZQogICAgICAgICkKICAgICAgICBzZWxmLnJlbHUgPSBubi5SZUxVKCkgaWYgcmVsdSBlbHNlIE5vbmUKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4ID0gc2VsZi5jb252KHgpCiAgICAgICAgaWYgc2VsZi5ibiBpcyBub3QgTm9uZToKICAg'
        'ICAgICAgICAgeCA9IHNlbGYuYm4oeCkKICAgICAgICBpZiBzZWxmLnJlbHUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHggPSBzZWxmLnJlbHUoeCkKICAgICAgICByZXR1cm4geAoKCmNsYXNzIENoYW5uZWxQb29sKG5uLk1vZHVsZSk6CiAgICBkZWYgZm9yd2Fy'
        'ZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gdG9yY2guY2F0KAogICAgICAgICAgICAodG9yY2gubWF4KHgsIDEpWzBdLnVuc3F1ZWV6ZSgxKSwgdG9yY2gubWVhbih4LCAxKS51bnNxdWVlemUoMSkpLCBkaW09MQogICAgICAgICkKCgpjbGFzcyBTcGF0aWFsR2F0'
        'ZShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHN1cGVyKFNwYXRpYWxHYXRlLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAga2VybmVsX3NpemUgPSA3CiAgICAgICAgc2VsZi5jb21wcmVzcyA9IENoYW5uZWxQb29sKCkKICAgICAg'
        'ICBzZWxmLnNwYXRpYWwgPSBCYXNpY0NvbnYoCiAgICAgICAgICAgIDIsIDEsIGtlcm5lbF9zaXplLCBzdHJpZGU9MSwgcGFkZGluZz0oa2VybmVsX3NpemUgLSAxKSAvLyAyLCByZWx1PUZhbHNlCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgog'
        'ICAgICAgIHhfY29tcHJlc3MgPSBzZWxmLmNvbXByZXNzKHgpCiAgICAgICAgeF9vdXQgPSBzZWxmLnNwYXRpYWwoeF9jb21wcmVzcykKICAgICAgICBzY2FsZSA9IHRvcmNoLnNpZ21vaWRfKHhfb3V0KQogICAgICAgIHJldHVybiB4ICogc2NhbGUKCmRlZiBhdXRv'
        'cGFkKGssIHA9Tm9uZSk6CiAgICBpZiBwIGlzIE5vbmU6CiAgICAgICAgcCA9IGsgLy8gMiBpZiBpc2luc3RhbmNlKGssIGludCkgZWxzZSBbeCAvLyAyIGZvciB4IGluIGtdIAogICAgcmV0dXJuIHAKICAgIApjbGFzcyBDb252KG5uLk1vZHVsZSk6CiAgICBkZWYg'
        'X19pbml0X18oc2VsZiwgYzEsIGMyLCBrPTEsIHM9MSwgcD1Ob25lLCBnPTEsIGFjdD1ubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpKTogICMgY2hfaW4sIGNoX291dCwga2VybmVsLCBzdHJpZGUsIHBhZGRpbmcsIGdyb3VwcwogICAgICAgIHN1cGVyKENv'
        'bnYsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYgICA9IG5uLkNvbnYyZChjMSwgYzIsIGssIHMsIGF1dG9wYWQoaywgcCksIGdyb3Vwcz1nLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4gICAgID0gbm4uQmF0Y2hOb3JtMmQoYzIsIGVwcz0w'
        'LjAwMSwgbW9tZW50dW09MC4wMykKICAgICAgICBzZWxmLmFjdCAgICA9IG5uLkxlYWt5UmVMVSgwLjEsIGlucGxhY2U9VHJ1ZSkgaWYgYWN0IGlzIFRydWUgZWxzZSAoYWN0IGlmIGlzaW5zdGFuY2UoYWN0LCBubi5Nb2R1bGUpIGVsc2Ugbm4uSWRlbnRpdHkoKSkK'
        'CgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2VsZi5jb252KHgpKSkKI2xpZ2h0aW5nIGRlaGF6ZSBuZXR3b3JrCmNsYXNzIExNRE5ldChubi5Nb2R1bGUpOgoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToK'
        'ICAgICAgICBzdXBlcihMTUROZXQsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICAjIG1haW5OZXQgQXJjaGl0ZWN0dXJlCiAgICAgICAgc2VsZi5BQU0gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoNjQsIDMsIDEsIDEpLAogICAgICAgICAg'
        'ICBubi5MZWFreVJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uRHJvcG91dCgwLjUpCiAgICAgICAgKQogICAgICAgIHNlbGYuQUFNXzEgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0n'
        'YmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQoMTI4LCAzMiwgMSwgMSksCiAgICAgICAgICAgIG5uLkxlYWt5UmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Ecm9wb3V0KDAuNSkKICAgICAgICApCiAg'
        'ICAgICAgc2VsZi5BQU1fMiA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLlVwc2FtcGxlKHNjYWxlX2ZhY3Rvcj00LCBtb2RlPSdiaWxpbmVhcicsIGFsaWduX2Nvcm5lcnM9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYyZCgyNTYsIDMyLCAxLCAxKSwK'
        'ICAgICAgICAgICAgbm4uTGVha3lSZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkRyb3BvdXQoMC41KQogICAgICAgICkKICAgICAgICBzZWxmLlRBID0gVHJpcGxldEF0dGVudGlvbig2NCkKCiAgICAgICAgc2VsZi5jb252ID0gQ29udig2NCwgMywg'
        'MywgMSkKCiAgICAgICAgc2VsZi51cDQgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9NCwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5yZWx1ID0gbm4uTGVha3lSZUxVKDAuMSwgaW5wbGFjZT1UcnVlKQoKCiAgICBk'
        'ZWYgZm9yd2FyZChzZWxmLCBmMSwgZjIsIGYzKToKICAgICAgICB0ID0gc2VsZi5BQU0oZjEpIAogICAgICAgIGYyID0gc2VsZi5BQU1fMShmMikKICAgICAgICBmMyA9IHNlbGYuQUFNXzIoZjMpCiAgICAgICAgeDEgPSBmMQogICAgICAgIHgyID0gdG9yY2guY2F0'
        'KFtmMiwgZjNdLCBkaW09MSkKICAgICAgICAKICAgICAgICB4ID0geDEgKyB4MgogICAgICAgIHggPSBzZWxmLlRBKHgpCiAgICAgICAgeCA9IHNlbGYuY29udih4KQoKICAgICAgICBkZWhhemUgPSAoKHggKiB0KSAtIHggKyAxKQoKICAgICAgICBvdXQgPSBzZWxm'
        'LnVwNChkZWhhemUpCiAgICAgICAgb3V0ID0gc2VsZi5yZWx1KG91dCkgICAKCiAgICAgICAgcmV0dXJuIG91dAogICAgICAgIApjbGFzcyBUcmlwbGV0QXR0ZW50aW9uKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBp'
        'bl9jaGFubmVscywKICAgICAgICByZWR1Y3Rpb25fcmF0aW89MTYsCiAgICAgICAgcG9vbF90eXBlcz1bImF2ZyIsICJtYXgiXSwKICAgICAgICBub19zcGF0aWFsPUZhbHNlLAogICAgKToKICAgICAgICBzdXBlcihUcmlwbGV0QXR0ZW50aW9uLCBzZWxmKS5fX2lu'
        'aXRfXygpCiAgICAgICAgc2VsZi5DaGFubmVsR2F0ZUggPSBTcGF0aWFsR2F0ZSgpCiAgICAgICAgc2VsZi5DaGFubmVsR2F0ZVcgPSBTcGF0aWFsR2F0ZSgpCiAgICAgICAgc2VsZi5ub19zcGF0aWFsID0gbm9fc3BhdGlhbAogICAgICAgIGlmIG5vdCBub19zcGF0'
        'aWFsOgogICAgICAgICAgICBzZWxmLlNwYXRpYWxHYXRlID0gU3BhdGlhbEdhdGUoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHhfcGVybTEgPSB4LnBlcm11dGUoMCwgMiwgMSwgMykuY29udGlndW91cygpCiAgICAgICAgeF9vdXQxID0gc2Vs'
        'Zi5DaGFubmVsR2F0ZUgoeF9wZXJtMSkKICAgICAgICB4X291dDExID0geF9vdXQxLnBlcm11dGUoMCwgMiwgMSwgMykuY29udGlndW91cygpCiAgICAgICAgeF9wZXJtMiA9IHgucGVybXV0ZSgwLCAzLCAyLCAxKS5jb250aWd1b3VzKCkKICAgICAgICB4X291dDIg'
        'PSBzZWxmLkNoYW5uZWxHYXRlVyh4X3Blcm0yKQogICAgICAgIHhfb3V0MjEgPSB4X291dDIucGVybXV0ZSgwLCAzLCAyLCAxKS5jb250aWd1b3VzKCkKICAgICAgICBpZiBub3Qgc2VsZi5ub19zcGF0aWFsOgogICAgICAgICAgICB4X291dCA9IHNlbGYuU3BhdGlh'
        'bEdhdGUoeCkKICAgICAgICAgICAgeF9vdXQgPSAoMSAvIDMpICogKHhfb3V0ICsgeF9vdXQxMSArIHhfb3V0MjEpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgeF9vdXQgPSAoMSAvIDIpICogKHhfb3V0MTEgKyB4X291dDIxKQogICAgICAgIHJldHVybiB4X291'
        'dAoKY2xhc3MgU2lMVShubi5Nb2R1bGUpOgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIGZvcndhcmQoeCk6CiAgICAgICAgcmV0dXJuIHggKiB0b3JjaC5zaWdtb2lkKHgpCgpkZWYgYXV0b3BhZChrLCBwPU5vbmUpOgogICAgaWYgcCBpcyBOb25lOgogICAgICAg'
        'IHAgPSBrIC8vIDIgaWYgaXNpbnN0YW5jZShrLCBpbnQpIGVsc2UgW3ggLy8gMiBmb3IgeCBpbiBrXSAKICAgIHJldHVybiBwCgpjbGFzcyBDb252KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYzEsIGMyLCBrPTEsIHM9MSwgcD1Ob25lLCBnPTEs'
        'IGFjdD1ubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpKTogIAogICAgICAgIHN1cGVyKENvbnYsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbnYgICA9IG5uLkNvbnYyZChjMSwgYzIsIGssIHMsIGF1dG9wYWQoaywgcCksIGdyb3Vwcz1nLCBi'
        'aWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4gICAgID0gbm4uQmF0Y2hOb3JtMmQoYzIsIGVwcz0wLjAwMSwgbW9tZW50dW09MC4wMykKICAgICAgICBzZWxmLmFjdCAgICA9IG5uLkxlYWt5UmVMVSgwLjEsIGlucGxhY2U9VHJ1ZSkgaWYgYWN0IGlzIFRydWUgZWxz'
        'ZSAoYWN0IGlmIGlzaW5zdGFuY2UoYWN0LCBubi5Nb2R1bGUpIGVsc2Ugbm4uSWRlbnRpdHkoKSkKCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2VsZi5jb252KHgpKSkKCiAgICBkZWYgZnVzZWZvcndh'
        'cmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuY29udih4KSkKICAgIApjbGFzcyBHSUUodG9yY2gubm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBlcHNpbG9uPTFlLTgpOgogICAgICAgIHN1cGVyKEdJRSwgc2VsZikuX19p'
        'bml0X18oKQogICAgICAgIHNlbGYuZXBzaWxvbiA9IGVwc2lsb24KCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICAjIFN0ZXAgMTogUGl4ZWwgTWVhbiBTcXVhcmVkCiAgICAgICAgeF9tZWFuID0gdG9yY2gubWVhbih4LCBkaW09KDIsIDMpLCBrZWVw'
        'ZGltPVRydWUpIAogICAgICAgIGVwc2lsb24gPSAoeCAtIHhfbWVhbikgKiogMiAgCiAgICAgICAgIyBTdGVwIDI6IEdsb2JhbCBFeHRyYWN0aW9uCiAgICAgICAgZXBzaWxvbl9tZWFuID0gdG9yY2gubWVhbihlcHNpbG9uLCBkaW09KDIsIDMpLCBrZWVwZGltPUZh'
        'bHNlKSAgCiAgICAgICAgZXBzaWxvbl9tZWFuICs9IHNlbGYuZXBzaWxvbgogICAgICAgICMgU3RlcCAzOiBHYW1tYSBDYWxjdWxhdGlvbiAoR2xvYmFsIEV4dHJhY3Rpb24pCiAgICAgICAgZ2FtbWFfdF9jID0gZXBzaWxvbiAvIGVwc2lsb25fbWVhbi51bnNxdWVl'
        'emUoLTEpLnVuc3F1ZWV6ZSgtMSkgIAogICAgICAgIHNpZ21vaWRfZ2FtbWEgPSB0b3JjaC5zaWdtb2lkKGdhbW1hX3RfYykgCiAgICAgICAgb3V0cHV0ID0geCAqIHNpZ21vaWRfZ2FtbWEgCiAgICAgICAgcmV0dXJuIG91dHB1dAogICAgCiMgTXVsdGktYnJhbmNo'
        'IFBvb2xpbmcgSW5mb3JtYXRpb24gRnVzaW9uIE1vZHVsZQpjbGFzcyBNUElGKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYzEsIGMyLCBjMywgcz0yLCBuPTQsIGU9MSwgaWRzPVswXSk6CiAgICAgICAgc3VwZXIoTVBJRiwgc2VsZikuX19pbml0'
        'X18oKQogICAgICAgIGNfID0gaW50KGMyICogZSkKICAgICAgICAKICAgICAgICBzZWxmLmlkcyA9IGlkcwogICAgICAgIGlmIHMgPT0gMToKICAgICAgICAgICAgc2VsZi5tMSA9IG5uLk1heFBvb2wyZChrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9cywgcGFkZGluZz0x'
        'KQogICAgICAgICAgICBzZWxmLm0yID0gbm4uQXZnUG9vbDJkKGtlcm5lbF9zaXplPTMsIHN0cmlkZT1zLCBwYWRkaW5nPTEpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5tMSA9IG5uLk1heFBvb2wyZChrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9cykKICAg'
        'ICAgICAgICAgc2VsZi5tMiA9IG5uLkF2Z1Bvb2wyZChrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9cykKICAgICAgICAKICAgICAgICBzZWxmLmN2MSA9IENvbnYoYzEsIGNfLCAxLCAxKQogICAgICAgIHNlbGYuY3YyID0gQ29udihjMSwgY18sIDEsIDEpCiAgICAgICAg'
        'c2VsZi5jdjMgPSBubi5Nb2R1bGVMaXN0KAogICAgICAgICAgICBbQ29udihjXyBpZiBpID09MCBlbHNlIGMyLCBjMiwgMywgMSkgZm9yIGkgaW4gcmFuZ2UobildCiAgICAgICAgKQogICAgICAgIHNlbGYuY3Y0ID0gQ29udihjXyAqIDIgKyBjMiAqIChsZW4oaWRz'
        'KSAtIDIpLCBjMywgMSwgMSkKCiAgICAgICAgc2VsZi5HSUUgPSBHSUUoYzEpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeDEgPSBzZWxmLm0xKHgpCiAgICAgICAgeDIgPSBzZWxmLm0yKHgpCiAgICAgICAgeCA9IHgxICsgeDIKICAgICAgICB4'
        'XzEgPSBzZWxmLmN2MSh4KQogICAgICAgIHhfMSA9IHNlbGYuR0lFKHhfMSkKICAgICAgICB4XzIgPSBzZWxmLmN2Mih4KQogICAgICAgIAogICAgICAgIHhfYWxsID0gW3hfMSwgeF8yXQoKICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4oc2VsZi5jdjMpKToKICAg'
        'ICAgICAgICAgeF8yID0gc2VsZi5jdjNbaV0oeF8yKQogICAgICAgICAgICB4X2FsbC5hcHBlbmQoeF8yKQogICAgICAgIAogICAgICAgIG91dCA9IHNlbGYuY3Y0KHRvcmNoLmNhdChbeF9hbGxbaWRdIGZvciBpZCBpbiBzZWxmLmlkc10sIDEpKQoKICAgICAgICBy'
        'ZXR1cm4gb3V0CiAgICAKY2xhc3MgRGlsYXRlZENvbnZOZXQobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFubmVscywgb3V0X2NoYW5uZWxzLCBkaWxhdGlvbiwgcGFkZGluZywga2VybmVsX3NpemUpOgogICAgICAgIHN1cGVyKERpbGF0'
        'ZWRDb252TmV0LCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5kaWxhdGVkX2NvbnYgPSBubi5Db252MmQoaW5fY2hhbm5lbHMsIG91dF9jaGFubmVscywga2VybmVsX3NpemU9a2VybmVsX3NpemUsIHN0cmlkZT0xLCBwYWRkaW5nPXBhZGRpbmcsIGRpbGF0'
        'aW9uPWRpbGF0aW9uKQogICAgICAgIHNlbGYucmVsdSA9IG5uLlJlTFUoaW5wbGFjZT1GYWxzZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKCiAgICAgICAgeCA9IHNlbGYuZGlsYXRlZF9jb252KHgpCiAgICAgICAgeCA9IHNlbGYucmVsdSh4KQoKICAgICAg'
        'ICByZXR1cm4geAogICAgCmNsYXNzIFNQUEVMQU4obm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjMSwgYzIsIGMzPTE2KTogIAogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYyA9IGMzCiAgICAgICAgc2VsZi5jdjEgPSBD'
        'b252KGMxLCBjMywgMSwgMSkKICAgICAgICBzZWxmLmN2MiA9IERpbGF0ZWRDb252TmV0KGMzLCBjMywga2VybmVsX3NpemU9MywgcGFkZGluZz0xLCBkaWxhdGlvbj0xKQogICAgICAgIHNlbGYuY3YzID0gRGlsYXRlZENvbnZOZXQoYzMsIGMzLCBrZXJuZWxfc2l6'
        'ZT0zLCBwYWRkaW5nPTIsIGRpbGF0aW9uPTIpCiAgICAgICAgc2VsZi5jdjQgPSBEaWxhdGVkQ29udk5ldChjMywgYzMsIGtlcm5lbF9zaXplPTMsIHBhZGRpbmc9MywgZGlsYXRpb249MykKICAgICAgICBzZWxmLmN2NSA9IENvbnYoNCpjMywgYzIsIDEsIDEpCgog'
        'ICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeSA9IFtzZWxmLmN2MSh4KV0KICAgICAgICB5LmV4dGVuZChtKHlbLTFdKSBmb3IgbSBpbiBbc2VsZi5jdjIsIHNlbGYuY3YzLCBzZWxmLmN2NF0pCiAgICAgICAgcmV0dXJuIHNlbGYuY3Y1KHRvcmNoLmNh'
        'dCh5LCAxKSkKICAgICAgICAKCgpjbGFzcyBFQ0Eobm4uTW9kdWxlKToKICAgICIiIkVmZmljaWVudCBDaGFubmVsIEF0dGVudGlvbiAoV2FuZyBldCBhbC4sIENWUFIgMjAyMCkgLS0gbmVhci16ZXJvLXBhcmFtZXRlcgogICAgY2hhbm5lbCBhdHRlbnRpb24gdmlh'
        'IGEgMUQgY29udm9sdXRpb24gb3ZlciBnbG9iYWxseSBwb29sZWQgY2hhbm5lbCBkZXNjcmlwdG9ycy4KICAgIFVzZWQgaW5zaWRlIERSTSB0byBjaGVhcGx5IHJlY2FsaWJyYXRlIHdoaWNoIGNoYW5uZWxzIG1hdHRlciBhZnRlciB0aGUKICAgIGRldGFpbC1yZWNv'
        'dmVyeSBicmFuY2hlcywgd2l0aG91dCB0aGUgY29zdCBvZiBhIGZ1bGwgc3F1ZWV6ZS1leGNpdGUgTUxQLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoYW5uZWxzLCBrX3NpemU9Myk6CiAgICAgICAgc3VwZXIoRUNBLCBzZWxmKS5fX2luaXRfXygpCiAgICAg'
        'ICAgc2VsZi5hdmdfcG9vbCA9IG5uLkFkYXB0aXZlQXZnUG9vbDJkKDEpCiAgICAgICAgc2VsZi5jb252ICAgICA9IG5uLkNvbnYxZCgxLCAxLCBrZXJuZWxfc2l6ZT1rX3NpemUsIHBhZGRpbmc9KGtfc2l6ZSAtIDEpIC8vIDIsIGJpYXM9RmFsc2UpCiAgICAgICAg'
        'c2VsZi5zaWdtb2lkICA9IG5uLlNpZ21vaWQoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHkgPSBzZWxmLmF2Z19wb29sKHgpICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIChCLCBDLCAxLCAxKQogICAgICAgIHkgPSB5LnNxdWVlemUo'
        'LTEpLnRyYW5zcG9zZSgtMSwgLTIpICAgICAgICAgICAgICAgICMgKEIsIDEsIEMpCiAgICAgICAgeSA9IHNlbGYuY29udih5KSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoQiwgMSwgQykKICAgICAgICB5ID0geS50cmFuc3Bvc2UoLTEsIC0y'
        'KS51bnNxdWVlemUoLTEpICAgICAgICAgICAgICAgIyAoQiwgQywgMSwgMSkKICAgICAgICB5ID0gc2VsZi5zaWdtb2lkKHkpCiAgICAgICAgcmV0dXJuIHggKiB5CgoKY2xhc3MgRGVwdGh3aXNlRGlsYXRlZENvbnYobm4uTW9kdWxlKToKICAgICIiIkNoZWFwIGRl'
        'cHRod2lzZSBkaWxhdGVkIDN4MyBjb252ICsgQk4gKyBhY3RpdmF0aW9uLiBEZXB0aHdpc2Uga2VlcHMgdGhpcwogICAgYWZmb3JkYWJsZSBhdCBQMydzIGhpZ2ggc3BhdGlhbCByZXNvbHV0aW9uOyB0aGUgZGlsYXRpb24gcmF0ZSBjb250cm9scyBob3cKICAgIG11'
        'Y2ggbG9jYWwgY29udGV4dCBlYWNoIGJyYW5jaCBvZiBEUk0gZ2F0aGVycy4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjaGFubmVscywgZGlsYXRpb24pOgogICAgICAgIHN1cGVyKERlcHRod2lzZURpbGF0ZWRDb252LCBzZWxmKS5fX2luaXRfXygpCiAgICAg'
        'ICAgc2VsZi5kdyAgPSBubi5Db252MmQoY2hhbm5lbHMsIGNoYW5uZWxzLCBrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9MSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFkZGluZz1kaWxhdGlvbiwgZGlsYXRpb249ZGlsYXRpb24sIGdyb3Vwcz1jaGFubmVs'
        'cywgYmlhcz1GYWxzZSkKICAgICAgICBzZWxmLmJuICA9IG5uLkJhdGNoTm9ybTJkKGNoYW5uZWxzLCBlcHM9MC4wMDEsIG1vbWVudHVtPTAuMDMpCiAgICAgICAgc2VsZi5hY3QgPSBubi5MZWFreVJlTFUoMC4xLCBpbnBsYWNlPVRydWUpCgogICAgZGVmIGZvcndh'
        'cmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYWN0KHNlbGYuYm4oc2VsZi5kdyh4KSkpCgoKY2xhc3MgRFJNKG5uLk1vZHVsZSk6CiAgICAiIiJEZXRhaWwgUmVjb3ZlcnkgTW9kdWxlLgoKICAgIFJERk5ldCdzIFJGRSBtb2R1bGUgZW5sYXJnZXMgdGhl'
        'IHJlY2VwdGl2ZSBmaWVsZCBvbmx5IGF0IHRoZSBjb2Fyc2UgUDUvU1BQCiAgICBzdGFnZS4gU21hbGwvdGhpbiBvYmplY3RzIChlLmcuIGJpY3ljbGUgLS0gdGhlIHdlYWtlc3QgY2xhc3Mgb24gYm90aAogICAgVk9DLUZPRyBhbmQgUlRUUyBpbiB0aGUgYmFzZSBw'
        'YXBlcidzIG93biBwZXItY2xhc3MgcmVzdWx0cykgYXJlIGRldGVjdGVkCiAgICBhdCB0aGUgZmluZXIgUDMgc2NhbGUsIHdoaWNoIHJlY2VpdmVzIG5vIGVxdWl2YWxlbnQgZGV0YWlsLXJlY292ZXJ5CiAgICB0cmVhdG1lbnQsIGFuZCBmb2cgc3BlY2lmaWNhbGx5'
        'IGF0dGVudWF0ZXMgdGhlIGhpZ2gtZnJlcXVlbmN5IGRldGFpbCBzdWNoCiAgICBvYmplY3RzIGRlcGVuZCBvbi4gRFJNIGlzIGEgbGlnaHR3ZWlnaHQsIHJlc2lkdWFsIHNpYmxpbmcgb2YgUkZFIGFwcGxpZWQKICAgIGF0IFAzOiB0d28gZGVwdGh3aXNlIGRpbGF0'
        'ZWQtY29udiBicmFuY2hlcyAoZGlsYXRpb24gMSBhbmQgMikgcmVjb3ZlcgogICAgbG9jYWwgZGV0YWlsIGF0IGxvdyBjaGFubmVsIHdpZHRoLCBhbiBFQ0EgYmxvY2sgcmVjYWxpYnJhdGVzIGNoYW5uZWxzLAogICAgYW5kIGEgMXgxIHByb2plY3Rpb24gZnVzZXMg'
        'YmFjayB0byB0aGUgb3JpZ2luYWwgY2hhbm5lbCBjb3VudC4KCiAgICBJbXBsZW1lbnRlZCBhcyBhIHplcm8taW5pdGlhbGl6ZWQgcmVzaWR1YWwgYnJhbmNoIC0tCiAgICAgICAgUDNfb3V0ID0gUDMgKyBmKFAzKQogICAgLS0gd2l0aCBmJ3MgZmluYWwgcHJvamVj'
        'dGlvbiB3ZWlnaHQgYW5kIGJpYXMgaW5pdGlhbGl6ZWQgdG8gemVybywgc28gdGhlCiAgICBtb2R1bGUgaXMgYW4gZXhhY3QgaWRlbnRpdHkgZnVuY3Rpb24gYXQgdGhlIHN0YXJ0IG9mIGZpbmUtdHVuaW5nIGZyb20gYQogICAgY29udmVyZ2VkIGNoZWNrcG9pbnQu'
        'IEl0IGNhbiBvbmx5IGJlZ2luIGNvbnRyaWJ1dGluZyBhcyB0cmFpbmluZyBtb3ZlcwogICAgdGhvc2Ugd2VpZ2h0cyBhd2F5IGZyb20gemVybywgd2hpY2ggYm91bmRzIHRoZSBkb3duc2lkZSBpZiB0aGUgZmluZS10dW5lCiAgICBidWRnZXQgdHVybnMgb3V0IHRv'
        'IGJlIHRvbyBzaG9ydCBmb3IgaXQgdG8gbGVhcm4gc29tZXRoaW5nIHVzZWZ1bC4KICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoYW5uZWxzLCByZWR1Y3Rpb249Mik6CiAgICAgICAgc3VwZXIoRFJNLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAgaGlk'
        'ZGVuID0gbWF4KGNoYW5uZWxzIC8vIHJlZHVjdGlvbiwgOCkKICAgICAgICBzZWxmLnJlZHVjZSAgPSBDb252KGNoYW5uZWxzLCBoaWRkZW4sIDEsIDEpCiAgICAgICAgc2VsZi5icmFuY2gxID0gRGVwdGh3aXNlRGlsYXRlZENvbnYoaGlkZGVuLCBkaWxhdGlvbj0x'
        'KQogICAgICAgIHNlbGYuYnJhbmNoMiA9IERlcHRod2lzZURpbGF0ZWRDb252KGhpZGRlbiwgZGlsYXRpb249MikKICAgICAgICBzZWxmLmVjYSAgICAgPSBFQ0EoaGlkZGVuICogMikKICAgICAgICBzZWxmLnByb2plY3QgPSBubi5Db252MmQoaGlkZGVuICogMiwg'
        'Y2hhbm5lbHMsIGtlcm5lbF9zaXplPTEsIHN0cmlkZT0xLCBiaWFzPVRydWUpCgogICAgICAgICMgWmVyby1pbml0OiB0aGlzIGJyYW5jaCBtdXN0IHN0YXJ0IGFzIGFuIGlkZW50aXR5IGZ1bmN0aW9uIHNvIGl0CiAgICAgICAgIyBjYW5ub3QgZGVzdGFiaWxpemUg'
        'YW4gYWxyZWFkeS1jb252ZXJnZWQgY2hlY2twb2ludCBvbiBlcG9jaCAwLgogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYucHJvamVjdC53ZWlnaHQpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5wcm9qZWN0LmJpYXMpCgogICAgZGVmIGZvcndhcmQoc2Vs'
        'ZiwgeCk6CiAgICAgICAgeSAgPSBzZWxmLnJlZHVjZSh4KQogICAgICAgIHkxID0gc2VsZi5icmFuY2gxKHkpCiAgICAgICAgeTIgPSBzZWxmLmJyYW5jaDIoeSkKICAgICAgICB5ICA9IHRvcmNoLmNhdChbeTEsIHkyXSwgZGltPTEpCiAgICAgICAgeSAgPSBzZWxm'
        'LmVjYSh5KQogICAgICAgIHkgID0gc2VsZi5wcm9qZWN0KHkpCiAgICAgICAgcmV0dXJuIHggKyB5CgoKZGVmIHByaW50X21vZGVsX2Zsb3BzX2FuZF9wYXJhbXMobW9kZWwsIGlucHV0cyk6CiAgICBmbG9wcywgcGFyYW1zID0gcHJvZmlsZShtb2RlbCwgaW5wdXRz'
        'PWlucHV0cykKICAgIHByaW50KGYiRkxPUHM6IHtmbG9wcyAvIDFlOTouMmZ9IEdGTE9QcyIpCiAgICBwcmludChmIlBhcmFtZXRlcnM6IHtwYXJhbXMgLyAxZTY6LjJmfSBNIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZmVhdDEgPSB0b3JjaC5y'
        'YW5kbigxLCA2NCwgMTYwLCAxNjApCiAgICBmZWF0MiA9IHRvcmNoLnJhbmRuKDEsIDEyOCwgODAsIDgwKSAKICAgIGZlYXQzID0gdG9yY2gucmFuZG4oMSwgMjU2LCA0MCwgNDApICAKICAgIGVuY29kZXIgPSBMTUROZXQoKQogICAgcHJpbnRfbW9kZWxfZmxvcHNf'
        'YW5kX3BhcmFtcyhlbmNvZGVyLCAoZmVhdDEsIGZlYXQyLCBmZWF0MykpCiAgICAK'
    ),
    'nets/model.py': (
        'aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIG5ldHMuQ29tbW9uIGltcG9ydCBDb252LCBTUFBFTEFOLCBEUk0KZnJvbSBuZXRzLmJhY2tib25lIGltcG9ydCBCYWNrYm9uZSwgTXVsdGlfQ29uY2F0X0Jsb2NrCnRyeToKICAgIGZyb20gY29u'
        'ZmlnIGltcG9ydCBVU0VfRFJNCmV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgIFVTRV9EUk0gPSBGYWxzZQoKZGVmIGZ1c2VfY29udl9hbmRfYm4oY29udiwgYm4pOgogICAgZnVzZWRjb252ID0gbm4uQ29udjJkKGNvbnYuaW5fY2hhbm5lbHMsCiAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgY29udi5vdXRfY2hhbm5lbHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAga2VybmVsX3NpemU9Y29udi5rZXJuZWxfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICBzdHJpZGU9Y29udi5zdHJpZGUsCiAgICAgICAgICAgICAgICAg'
        'ICAgICAgICAgcGFkZGluZz1jb252LnBhZGRpbmcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZ3JvdXBzPWNvbnYuZ3JvdXBzLAogICAgICAgICAgICAgICAgICAgICAgICAgIGJpYXM9VHJ1ZSkucmVxdWlyZXNfZ3JhZF8oRmFsc2UpLnRvKGNvbnYud2VpZ2h0'
        'LmRldmljZSkKCiAgICB3X2NvbnYgID0gY29udi53ZWlnaHQuY2xvbmUoKS52aWV3KGNvbnYub3V0X2NoYW5uZWxzLCAtMSkKICAgIHdfYm4gICAgPSB0b3JjaC5kaWFnKGJuLndlaWdodC5kaXYodG9yY2guc3FydChibi5lcHMgKyBibi5ydW5uaW5nX3ZhcikpKQog'
        'ICAgZnVzZWRjb252LndlaWdodC5jb3B5Xyh0b3JjaC5tbSh3X2JuLCB3X2NvbnYpLnZpZXcoZnVzZWRjb252LndlaWdodC5zaGFwZSkuZGV0YWNoKCkpCgogICAgYl9jb252ICA9IHRvcmNoLnplcm9zKGNvbnYud2VpZ2h0LnNpemUoMCksIGRldmljZT1jb252Lndl'
        'aWdodC5kZXZpY2UpIGlmIGNvbnYuYmlhcyBpcyBOb25lIGVsc2UgY29udi5iaWFzCiAgICBiX2JuICAgID0gYm4uYmlhcyAtIGJuLndlaWdodC5tdWwoYm4ucnVubmluZ19tZWFuKS5kaXYodG9yY2guc3FydChibi5ydW5uaW5nX3ZhciArIGJuLmVwcykpCiAgICBm'
        'dXNlZGNvbnYuYmlhcy5jb3B5XygodG9yY2gubW0od19ibiwgYl9jb252LnJlc2hhcGUoLTEsIDEpKS5yZXNoYXBlKC0xKSArIGJfYm4pLmRldGFjaCgpKQogICAgcmV0dXJuIGZ1c2VkY29udgoKY2xhc3MgTVAobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhz'
        'ZWxmLCBrPTIpOgogICAgICAgIHN1cGVyKE1QLCBzZWxmKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5tMSA9IG5uLk1heFBvb2wyZChrZXJuZWxfc2l6ZT1rLCBzdHJpZGU9aykKICAgICAgICBzZWxmLm0yID0gbm4uQXZnUG9vbDJkKGtlcm5lbF9zaXplPWssIHN0'
        'cmlkZT1rKQogICAgICAgIHNlbGYudXAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4MSA9IHNlbGYubTEoeCkKICAgICAgICB4MiA9IHNlbGYubTIoeCkKICAgICAgICByZXR1cm4gc2VsZi51'
        'cCh4MSArIHgyKQogICAgCmNsYXNzIFlvbG9Cb2R5KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYW5jaG9yc19tYXNrLCBudW1fY2xhc3Nlcyk6CiAgICAgICAgc3VwZXIoWW9sb0JvZHksIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICB0cmFuc2l0'
        'aW9uX2NoYW5uZWxzID0gMTYKICAgICAgICBibG9ja19jaGFubmVscyAgICAgID0gMTYKICAgICAgICBwYW5ldF9jaGFubmVscyAgICAgID0gMTYKICAgICAgICBlICAgICAgICAgICAgICAgICAgID0gMQogICAgICAgIG4gICAgICAgICAgICAgICAgICAgPSAyCiAg'
        'ICAgICAgaWRzICAgICAgICAgICAgICAgICA9IFstMSwgLTIsIC0zLCAtNF0KCiAgICAgICAgc2VsZi5iYWNrYm9uZSAgID0gQmFja2JvbmUodHJhbnNpdGlvbl9jaGFubmVscywgYmxvY2tfY2hhbm5lbHMsIG4pCgogICAgICAgIHNlbGYudXBzYW1wbGUgICA9IG5u'
        'LlVwc2FtcGxlKHNjYWxlX2ZhY3Rvcj0yLCBtb2RlPSJuZWFyZXN0IikKCiAgICAgICAgc2VsZi5zcHBlbGFuICAgICAgICAgICAgICAgID0gU1BQRUxBTih0cmFuc2l0aW9uX2NoYW5uZWxzICogMzIsIHRyYW5zaXRpb25fY2hhbm5lbHMgKiAxNikKICAgICAgICBz'
        'ZWxmLmNvbnZfZm9yX1A1ICAgICAgICAgICAgPSBDb252KHRyYW5zaXRpb25fY2hhbm5lbHMgKiAxNiwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDgpCiAgICAgICAgc2VsZi5jb252X2Zvcl9mZWF0MiAgICAgICAgID0gQ29udih0cmFuc2l0aW9uX2NoYW5uZWxzICog'
        'MTYsIHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4KQogICAgICAgIHNlbGYuY29udjNfZm9yX3Vwc2FtcGxlMSAgICA9IE11bHRpX0NvbmNhdF9CbG9jayh0cmFuc2l0aW9uX2NoYW5uZWxzICogMTYsIHBhbmV0X2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl9jaGFubmVs'
        'cyAqIDgsIGU9ZSwgbj1uLCBpZHM9aWRzKQoKICAgICAgICBzZWxmLmNvbnZfZm9yX1A0ICAgICAgICAgICAgPSBDb252KHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogNCkKICAgICAgICBzZWxmLmNvbnZfZm9yX2ZlYXQxICAg'
        'ICAgICAgPSBDb252KHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogNCkKICAgICAgICBzZWxmLmNvbnYzX2Zvcl91cHNhbXBsZTIgICAgPSBNdWx0aV9Db25jYXRfQmxvY2sodHJhbnNpdGlvbl9jaGFubmVscyAqIDgsIHBhbmV0'
        'X2NoYW5uZWxzICogMiwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDQsIGU9ZSwgbj1uLCBpZHM9aWRzKQoKICAgICAgICAjIERldGFpbCBSZWNvdmVyeSBNb2R1bGU6IHplcm8taW5pdCByZXNpZHVhbCBkZXRhaWwvYXR0ZW50aW9uIGJsb2NrIGF0CiAgICAgICAgIyB0'
        'aGUgZmluZXN0IChQMykgc2NhbGUsIHRhcmdldGluZyB0aGUgc21hbGwvdGhpbi1vYmplY3Qgd2Vha25lc3MKICAgICAgICAjIChiaWN5Y2xlLCBtb3RvcmJpa2UpIHRoYXQgUkRGTmV0J3MgUkZFIC0tIGFwcGxpZWQgb25seSBhdCB0aGUgY29hcnNlCiAgICAgICAg'
        'IyBQNS9TUFAgc3RhZ2UgLS0gZG9lcyBub3QgYWRkcmVzcy4gRmFsbHMgYmFjayB0byBhIG5vLW9wIGlkZW50aXR5CiAgICAgICAgIyB3aGVuIGNvbmZpZy5VU0VfRFJNIGlzIEZhbHNlLCBzbyB0aGUgImNvbnRyb2wiIGFyY2hpdGVjdHVyZSB1c2VkIHRvCiAgICAg'
        'ICAgIyBpc29sYXRlIHRoaXMgbW9kdWxlJ3MgY29udHJpYnV0aW9uIGlzIGJ5dGUtZm9yLWJ5dGUgdGhlIGJhc2UgcGFwZXIncy4KICAgICAgICBzZWxmLmRybSA9IERSTSh0cmFuc2l0aW9uX2NoYW5uZWxzICogNCkgaWYgVVNFX0RSTSBlbHNlIG5uLklkZW50aXR5'
        'KCkKCiAgICAgICAgc2VsZi5kb3duX3NhbXBsZTEgICAgICAgICAgID0gQ29udih0cmFuc2l0aW9uX2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDgsIGs9Mywgcz0yKQogICAgICAgIHNlbGYuY29udjNfZm9yX2Rvd25zYW1wbGUxICA9IE11bHRp'
        'X0NvbmNhdF9CbG9jayh0cmFuc2l0aW9uX2NoYW5uZWxzICogMTYsIHBhbmV0X2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDgsIGU9ZSwgbj1uLCBpZHM9aWRzKQoKICAgICAgICBzZWxmLmRvd25fc2FtcGxlMiAgICAgICAgICAgPSBDb252KHRy'
        'YW5zaXRpb25fY2hhbm5lbHMgKiA4LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogMTYsIGs9Mywgcz0yKQogICAgICAgIHNlbGYuY29udjNfZm9yX2Rvd25zYW1wbGUyICA9IE11bHRpX0NvbmNhdF9CbG9jayh0cmFuc2l0aW9uX2NoYW5uZWxzICogMzIsIHBhbmV0X2No'
        'YW5uZWxzICogOCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDE2LCBlPWUsIG49biwgaWRzPWlkcykKCiAgICAgICAgc2VsZi5wZiA9IE1QKCkKCiAgICAgICAgc2VsZi5yZXBfY29udl8xID0gQ29udih0cmFuc2l0aW9uX2NoYW5uZWxzICogNCwgdHJhbnNpdGlvbl9j'
        'aGFubmVscyAqIDgsIDMsIDEpCiAgICAgICAgc2VsZi5yZXBfY29udl8yID0gQ29udih0cmFuc2l0aW9uX2NoYW5uZWxzICogOCwgdHJhbnNpdGlvbl9jaGFubmVscyAqIDE2LCAzLCAxKQogICAgICAgIHNlbGYucmVwX2NvbnZfMyA9IENvbnYodHJhbnNpdGlvbl9j'
        'aGFubmVscyAqIDE2LCB0cmFuc2l0aW9uX2NoYW5uZWxzICogMzIsIDMsIDEpCgogICAgICAgIHNlbGYueW9sb19oZWFkX1AzID0gbm4uQ29udjJkKHRyYW5zaXRpb25fY2hhbm5lbHMgKiA4LCBsZW4oYW5jaG9yc19tYXNrWzJdKSAqICg1ICsgbnVtX2NsYXNzZXMp'
        'LCAxKQogICAgICAgIHNlbGYueW9sb19oZWFkX1A0ID0gbm4uQ29udjJkKHRyYW5zaXRpb25fY2hhbm5lbHMgKiAxNiwgbGVuKGFuY2hvcnNfbWFza1sxXSkgKiAoNSArIG51bV9jbGFzc2VzKSwgMSkKICAgICAgICBzZWxmLnlvbG9faGVhZF9QNSA9IG5uLkNvbnYy'
        'ZCh0cmFuc2l0aW9uX2NoYW5uZWxzICogMzIsIGxlbihhbmNob3JzX21hc2tbMF0pICogKDUgKyBudW1fY2xhc3NlcyksIDEpCgogICAgZGVmIGZ1c2Uoc2VsZik6CiAgICAgICAgcHJpbnQoJ0Z1c2luZyBsYXllcnMuLi4gJykKICAgICAgICBmb3IgbSBpbiBzZWxm'
        'Lm1vZHVsZXMoKToKICAgICAgICAgICAgaWYgdHlwZShtKSBpcyBDb252IGFuZCBoYXNhdHRyKG0sICdibicpOgogICAgICAgICAgICAgICAgbS5jb252ID0gZnVzZV9jb252X2FuZF9ibihtLmNvbnYsIG0uYm4pCiAgICAgICAgICAgICAgICBkZWxhdHRyKG0sICdi'
        'bicpCiAgICAgICAgICAgICAgICBtLmZvcndhcmQgPSBtLmZ1c2Vmb3J3YXJkCiAgICAgICAgcmV0dXJuIHNlbGYKICAgIAogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgaWYgc2VsZi50cmFpbmluZzoKICAgICAgICAgICAgZmVhdDEsIGZlYXQyLCBm'
        'ZWF0MywgZGVoYXppbmcgPSBzZWxmLmJhY2tib25lLmZvcndhcmQoeCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBmZWF0MSwgZmVhdDIsIGZlYXQzID0gc2VsZi5iYWNrYm9uZS5mb3J3YXJkKHgpCgogICAgICAgIFA1ICAgICAgICAgID0gc2VsZi5zcHBlbGFu'
        'KGZlYXQzKQogICAgICAgIFA1X2NvbnYgICAgID0gc2VsZi5jb252X2Zvcl9QNShQNSkKICAgICAgICBQNV91cHNhbXBsZSA9IHNlbGYudXBzYW1wbGUoUDVfY29udikKICAgICAgICBQNCAgICAgICAgICA9IHRvcmNoLmNhdChbc2VsZi5jb252X2Zvcl9mZWF0Mihm'
        'ZWF0MiksIFA1X3Vwc2FtcGxlXSwgMSkKICAgICAgICBQNCAgICAgICAgICA9IHNlbGYuY29udjNfZm9yX3Vwc2FtcGxlMShQNCkKCiAgICAgICAgUDRfY29udiAgICAgPSBzZWxmLmNvbnZfZm9yX1A0KFA0KQogICAgICAgIFA0X3Vwc2FtcGxlID0gc2VsZi51cHNh'
        'bXBsZShQNF9jb252KQogICAgICAgIFAzICAgICAgICAgID0gdG9yY2guY2F0KFtzZWxmLmNvbnZfZm9yX2ZlYXQxKGZlYXQxKSwgUDRfdXBzYW1wbGVdLCAxKQogICAgICAgIFAzICAgICAgICAgID0gc2VsZi5jb252M19mb3JfdXBzYW1wbGUyKFAzKQogICAgICAg'
        'IFAzICAgICAgICAgID0gc2VsZi5kcm0oUDMpCgogICAgICAgIFAzX2Rvd25zYW1wbGUgPSBzZWxmLmRvd25fc2FtcGxlMShQMykKICAgICAgICBQNCA9IHRvcmNoLmNhdChbUDNfZG93bnNhbXBsZSwgUDRdLCAxKQogICAgICAgIFA0ID0gc2VsZi5jb252M19mb3Jf'
        'ZG93bnNhbXBsZTEoUDQpCiAgICAgICAgUDQgPSBzZWxmLnBmKFA0KQoKICAgICAgICBQNF9kb3duc2FtcGxlID0gc2VsZi5kb3duX3NhbXBsZTIoUDQpCiAgICAgICAgUDUgPSB0b3JjaC5jYXQoW1A0X2Rvd25zYW1wbGUsIFA1XSwgMSkKICAgICAgICBQNSA9IHNl'
        'bGYuY29udjNfZm9yX2Rvd25zYW1wbGUyKFA1KQogICAgICAgIAogICAgICAgIFAzID0gc2VsZi5yZXBfY29udl8xKFAzKQogICAgICAgIFA0ID0gc2VsZi5yZXBfY29udl8yKFA0KQogICAgICAgIFA1ID0gc2VsZi5yZXBfY29udl8zKFA1KQoKICAgICAgICBvdXQy'
        'ID0gc2VsZi55b2xvX2hlYWRfUDMoUDMpCiAgICAgICAgb3V0MSA9IHNlbGYueW9sb19oZWFkX1A0KFA0KQogICAgICAgIG91dDAgPSBzZWxmLnlvbG9faGVhZF9QNShQNSkKCiAgICAgICAgaWYgc2VsZi50cmFpbmluZzoKICAgICAgICAgICAgcmV0dXJuIFtvdXQw'
        'LCBvdXQxLCBvdXQyLCBkZWhhemluZ10KICAgICAgICBlbHNlOgogICAgICAgICAgICByZXR1cm4gW291dDAsIG91dDEsIG91dDJd'
    ),
    'config.py': (
        'Q3VkYSAgICAgICAgICAgID0gVHJ1ZQpzZWVkICAgICAgICAgICAgPSAxMTQ1MTQKZGlzdHJpYnV0ZWQgICAgID0gRmFsc2UKc3luY19ibiAgICAgICAgID0gRmFsc2UKZnAxNiAgICAgICAgICAgID0gRmFsc2UKY2xhc3Nlc19wYXRoICAgID0gJ21vZGVsX2RhdGEv'
        'cnR0c19jbGFzc2VzLnR4dCcKYW5jaG9yc19wYXRoICAgID0gJ21vZGVsX2RhdGEveW9sb19hbmNob3JzLnR4dCcKYW5jaG9yc19tYXNrICAgID0gW1s2LCA3LCA4XSwgWzMsIDQsIDVdLCBbMCwgMSwgMl1dCm1vZGVsX3BhdGggICAgICA9ICdtb2RlbF9kYXRhL3lv'
        'bG92N190aW55X3dlaWdodHMucHRoJwppbnB1dF9zaGFwZSAgICAgPSBbNjQwLCA2NDBdCnByZXRyYWluZWQgICAgICA9IEZhbHNlCkluaXRfRXBvY2ggICAgICAgICAgPSAwCkZyZWV6ZV9FcG9jaCAgICAgICAgPSAxMDAKRnJlZXplX2JhdGNoX3NpemUgICA9IDE2'
        'ClVuRnJlZXplX0Vwb2NoICAgICAgPSAzMDAKVW5mcmVlemVfYmF0Y2hfc2l6ZSA9IDE2CkZyZWV6ZV9UcmFpbiAgICAgICAgPSBUcnVlCkluaXRfbHIgICAgICAgICAgICAgPSAxZS0yCk1pbl9sciAgICAgICAgICAgICAgPSBJbml0X2xyICogMC4wMQpvcHRpbWl6'
        'ZXJfdHlwZSAgICAgID0gInNnZCIKbW9tZW50dW0gICAgICAgICAgICA9IDAuOTM3CndlaWdodF9kZWNheSAgICAgICAgPSA1ZS00CmxyX2RlY2F5X3R5cGUgICAgICAgPSAiY29zIgpzYXZlX3BlcmlvZCAgICAgICAgID0gMTAKc2F2ZV9kaXIgICAgICAgICAgICA9'
        'ICdsb2dzJwpldmFsX2ZsYWcgICAgICAgICAgID0gVHJ1ZQpldmFsX3BlcmlvZCAgICAgICAgID0gMTAKbnVtX3dvcmtlcnMgICAgICAgICA9IDAKdHJhaW5fYW5ub3RhdGlvbl9wYXRoICAgPSAnMjAwN190cmFpbl9mb2cudHh0Jwp2YWxfYW5ub3RhdGlvbl9wYXRo'
        'ICAgICA9ICcyMDA3X3ZhbF9mb2cudHh0JwpjbGVhcl9hbm5vdGF0aW9uX3BhdGggPSAnMjAwN190cmFpbi50eHQnCgojIC0tLSBUaGVzaXMgbW9kaWZpY2F0aW9ucyBvdmVyIHRoZSBiYXNlIHBhcGVyLCBhbGwgZGVmYXVsdCBPRkYgc28gdGhlIC0tLQojIC0tLSBj'
        'b21taXR0ZWQgcmVwbyByZXByb2R1Y2VzIHRoZSBiYXNlIHBhcGVyJ3MgZXhhY3QgYmVoYXZpb3IgdW5sZXNzICAgLS0tCiMgLS0tIGV4cGxpY2l0bHkgZW5hYmxlZCAoZS5nLiBieSB0aGUgZmluZS10dW5pbmcgbm90ZWJvb2spLiAgICAgICAgICAgICAtLS0KIyBE'
        'Uk06IHplcm8taW5pdCByZXNpZHVhbCBEZXRhaWwgUmVjb3ZlcnkgTW9kdWxlIGF0IHRoZSBQMyAoZmluZXN0KSBzY2FsZSwKIyAgICAgIHRhcmdldGluZyB0aGUgc21hbGwvdGhpbi1vYmplY3Qgd2Vha25lc3MgKGJpY3ljbGUsIG1vdG9yYmlrZSkuClVTRV9EUk0g'
        'ICAgICAgICA9IEZhbHNlCiMgV2lzZS1Jb1U6IGR5bmFtaWMgbm9uLW1vbm90b25pYyBmb2N1c2luZyBhcHBsaWVkIG9uIHRvcCBvZiB0aGUgZXhpc3RpbmcKIyAgICAgICAgICAgQ0lvVSBib3gtcmVncmVzc2lvbiBsb3NzLgpVU0VfV0lTRV9JT1UgICAgPSBGYWxz'
        'ZQojIERXQTogRHluYW1pYyBXZWlnaHQgQXZlcmFnaW5nIChMaXUgZXQgYWwuLCBDVlBSIDIwMTkpIGZvciB0aGUgZGV0ZWN0aW9uIC8KIyAgICAgIGRlaGF6ZSB0YXNrIGJhbGFuY2UsIHJlcGxhY2luZyB0aGUgZml4ZWQgbGFtYmRhPTAuMSBiZWxvdy4KVVNFX0RX'
        'QSAgICAgICAgID0gRmFsc2U='
    ),
    'nets/yolo_training.py': (
        '...'
    ),
    'utils/utils_fit.py': (
        'aW1wb3J0IG9zCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KZnJvbSB0cWRtIGltcG9ydCB0cWRtCmZyb20gdXRpbHMudXRpbHMgaW1wb3J0IGdldF9scgoKZGVmIGZpdF9vbmVfZXBvY2gobW9kZWxfdHJhaW4sIG1vZGVsLCBlbWEsIHlvbG9fbG9z'
        'cywgbG9zc19oaXN0b3J5LCBldmFsX2NhbGxiYWNrLCBvcHRpbWl6ZXIsIGVwb2NoLCBlcG9jaF9zdGVwLCBnZW4sIEVwb2NoLCBjdWRhLCBmcDE2LCBzY2FsZXIsIHNhdmVfcGVyaW9kLCBzYXZlX2RpciwgbG9jYWxfcmFuaz0wKToKICAgIGxvc3MgICAgICAgID0g'
        'MAogICAgRGVoYXp5X2xvc3MgPSAwCiAgICBEZXRfbG9zcyAgICA9IDAKICAgIGNyaXRlcmlvbiA9IG5uLk1TRUxvc3MoKQoKICAgICMgRHluYW1pYyBXZWlnaHQgQXZlcmFnaW5nIChEV0EpIHRhc2sgd2VpZ2h0cywgcmVjb21wdXRlZCBvbmNlIHBlciBlcG9jaAog'
        'ICAgIyBmcm9tIHRoZSBwcmV2aW91cyB0d28gZXBvY2hzJyBhdmVyYWdlIGxvc3NlcyAoZXF1YWwgd2VpZ2h0cyB1bnRpbCB0aGVuLAogICAgIyBvciBhbHdheXMgZXF1YWwvZml4ZWQtbGFtYmRhLWNvbXBhdGlibGUgaWYgY29uZmlnLlVTRV9EV0EgaXMgRmFsc2Up'
        'LgogICAgd19kZXQsIHdfZGVoYXplID0geW9sb19sb3NzLmdldF90YXNrX3dlaWdodHMoKQoKICAgIGlmIGxvY2FsX3JhbmsgPT0gMDoKICAgICAgICBwcmludCgnU3RhcnQgVHJhaW4nKQogICAgICAgIHByaW50KCdUYXNrIHdlaWdodHMgdGhpcyBlcG9jaCAtPiBk'
        'ZXRlY3Rpb246ICUuM2YsIGRlaGF6ZTogJS4zZicgJSAod19kZXQsIHdfZGVoYXplKSkKICAgICAgICBwYmFyID0gdHFkbSh0b3RhbD1lcG9jaF9zdGVwLGRlc2M9ZidFcG9jaCB7ZXBvY2ggKyAxfS97RXBvY2h9Jyxwb3N0Zml4PWRpY3QsbWluaW50ZXJ2YWw9MC4z'
        'KQogICAgICAgIG1vZGVsX3RyYWluLnRyYWluKCkKCiAgICBmb3IgaXRlcmF0aW9uLCBiYXRjaCBpbiBlbnVtZXJhdGUoZ2VuKToKICAgICAgICBpZiBpdGVyYXRpb24gPj0gZXBvY2hfc3RlcDoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpbWFnZXMsIHRhcmdl'
        'dHMsIGNsZWFuID0gYmF0Y2hbMF0sIGJhdGNoWzFdLCBiYXRjaFsyXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBpZiBjdWRhOgogICAgICAgICAgICAgICAgaW1hZ2VzICA9IGltYWdlcy5jdWRhKGxvY2FsX3JhbmspCiAgICAgICAg'
        'ICAgICAgICB0YXJnZXRzID0gdGFyZ2V0cy5jdWRhKGxvY2FsX3JhbmspCiAgICAgICAgICAgICAgICBjbGVhbiA9IGNsZWFuLmN1ZGEobG9jYWxfcmFuaykKICAgICAgICAgICAgICAgIGhhenlfYW5kX2NsZWFyID0gdG9yY2guY2F0KFtpbWFnZXMsIGNsZWFuXSwg'
        'ZGltID0gMCkuY3VkYSgpCiAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpCgogICAgICAgIGlmIG5vdCBmcDE2OgogICAgICAgICAgICBvdXRwdXRzICAgICAgICAgPSBtb2RlbF90cmFpbihoYXp5X2FuZF9jbGVhcikKICAgICAgICAgICAgZGV0ZWN0X291dHB1'
        'dHMgPSBbb3V0cHV0c1swXSxvdXRwdXRzWzFdLG91dHB1dHNbMl1dCiAgICAgICAgICAgIGxvc3NfZGV0ZWN0aW9uICAgICAgPSB5b2xvX2xvc3MoZGV0ZWN0X291dHB1dHMsIHRhcmdldHMsIGltYWdlcykKICAgICAgICAgICAgbG9zc19kZWhhenkgICAgID0gY3Jp'
        'dGVyaW9uKG91dHB1dHNbM10sIGNsZWFuKQogICAgICAgICAgICBsb3NzX3ZhbHVlICAgICAgPSB3X2RldCAqIGxvc3NfZGV0ZWN0aW9uICsgMC4xICogd19kZWhhemUgKiBsb3NzX2RlaGF6eQogICAgICAgICAgICBsb3NzX3ZhbHVlLmJhY2t3YXJkKCkKICAgICAg'
        'ICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZyb20gdG9yY2guY3VkYS5hbXAgaW1wb3J0IGF1dG9jYXN0CiAgICAgICAgICAgIHdpdGggYXV0b2Nhc3QoKToKICAgICAgICAgICAgICAgIG91dHB1dHMgICAgICAgICA9IG1v'
        'ZGVsX3RyYWluKGltYWdlcykKICAgICAgICAgICAgICAgIGxvc3NfdmFsdWUgICAgICA9IHlvbG9fbG9zcyhvdXRwdXRzLCB0YXJnZXRzLCBpbWFnZXMpCiAgICAgICAgICAgIHNjYWxlci5zY2FsZShsb3NzX3ZhbHVlKS5iYWNrd2FyZCgpCiAgICAgICAgICAgIHNj'
        'YWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgc2NhbGVyLnVwZGF0ZSgpCiAgICAgICAgaWYgZW1hOgogICAgICAgICAgICBlbWEudXBkYXRlKG1vZGVsX3RyYWluKQogICAgICAgIERlaGF6eV9sb3NzICs9IGxvc3NfZGVoYXp5Lml0ZW0oKQogICAgICAg'
        'IERldF9sb3NzICAgICs9IGxvc3NfZGV0ZWN0aW9uLml0ZW0oKQogICAgICAgIGxvc3MgICAgICAgICs9IGxvc3NfdmFsdWUuaXRlbSgpCiAgICAgICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgICAgICBwYmFyLnNldF9wb3N0Zml4KCoqeydsb3NzJyAgOiBs'
        'b3NzIC8gKGl0ZXJhdGlvbiArIDEpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICdsb3NzX2RldGVjdGlvbicgIDogRGV0X2xvc3MgLyAoaXRlcmF0aW9uICsgMSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ0RlaGF6eV9sb3NzJzog'
        'RGVoYXp5X2xvc3MgLyAoaXRlcmF0aW9uICsgMSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgJ2xyJyAgICA6IGdldF9scihvcHRpbWl6ZXIpfSkKICAgICAgICAgICAgcGJhci51cGRhdGUoMSkKCiAgICAjIEZlZWQgdGhpcyBlcG9jaCdzIHJhdyAo'
        'dW53ZWlnaHRlZCkgdGFzayBsb3NzZXMgYmFjayBpbnRvIHRoZSBEV0EKICAgICMgaGlzdG9yeSBzbyBuZXh0IGVwb2NoJ3Mgd2VpZ2h0cyBjYW4gYmUgY29tcHV0ZWQuIEhhcm1sZXNzIG5vLW9wIHdoZW4KICAgICMgY29uZmlnLlVTRV9EV0EgaXMgRmFsc2UuCiAg'
        'ICB5b2xvX2xvc3MudXBkYXRlX3Rhc2tfbG9zc19oaXN0b3J5KERldF9sb3NzIC8gZXBvY2hfc3RlcCwgRGVoYXp5X2xvc3MgLyBlcG9jaF9zdGVwKQoKICAgIGlmIGVtYToKICAgICAgICBtb2RlbF90cmFpbl9ldmFsID0gZW1hLmVtYQogICAgZWxzZToKICAgICAg'
        'ICBtb2RlbF90cmFpbl9ldmFsID0gbW9kZWxfdHJhaW4uZXZhbCgpCgogICAgaWYgbG9jYWxfcmFuayA9PSAwOgogICAgICAgIHBiYXIuY2xvc2UoKQogICAgICAgIGxvc3NfaGlzdG9yeS5hcHBlbmRfbG9zcyhlcG9jaCArIDEsIGxvc3MgLyBlcG9jaF9zdGVwKQog'
        'ICAgICAgIGV2YWxfY2FsbGJhY2sub25fZXBvY2hfZW5kKGVwb2NoICsgMSwgbW9kZWxfdHJhaW5fZXZhbCkKICAgICAgICBwcmludCgnRXBvY2g6Jysgc3RyKGVwb2NoICsgMSkgKyAnLycgKyBzdHIoRXBvY2gpKQogICAgICAgIHByaW50KCdUb3RhbCBMb3NzOiAl'
        'LjNmJyAlIChsb3NzIC8gZXBvY2hfc3RlcCkpCiAgICAgICAgaWYgZW1hOgogICAgICAgICAgICBzYXZlX3N0YXRlX2RpY3QgPSBlbWEuZW1hLnN0YXRlX2RpY3QoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNhdmVfc3RhdGVfZGljdCA9IG1vZGVsLnN0YXRl'
        'X2RpY3QoKQogICAgICAgIGlmIChlcG9jaCArIDEpICUgc2F2ZV9wZXJpb2QgPT0gMCBvciBlcG9jaCArIDEgPT0gRXBvY2g6CiAgICAgICAgICAgIHRvcmNoLnNhdmUoc2F2ZV9zdGF0ZV9kaWN0LCBvcy5wYXRoLmpvaW4oc2F2ZV9kaXIsICJlcCUwM2QtbG9zcyUu'
        'M2YucHRoIiAlIChlcG9jaCArIDEsIGxvc3MgLyBlcG9jaF9zdGVwKSkpCgogICAgICAgICMgQmVzdC1jaGVja3BvaW50IHNlbGVjdGlvbjogcHJlZmVyIHZhbGlkYXRpb24gbUFQICh3aGF0IHdlIGFjdHVhbGx5CiAgICAgICAgIyBjYXJlIGFib3V0KSBvbmNlIGl0'
        'J3MgYXZhaWxhYmxlLCBzaW5jZSB0cmFpbiBsb3NzIHN0b3BzIGJlaW5nIGEKICAgICAgICAjIHJlbGlhYmxlIHByb3h5IGZvciBhY2N1cmFjeSB3aGVuIERXQSBpcyByZXdlaWdodGluZyB0aGUgbG9zcwogICAgICAgICMgZm9ybXVsYXRpb24gZXBvY2ggdG8gZXBv'
        'Y2guIEZhbGxzIGJhY2sgdG8gdHJhaW4gbG9zcyBvbmx5IG9uIHRoZQogICAgICAgICMgZXBvY2hzIGJlZm9yZSB0aGUgZmlyc3QgbUFQIGV2YWx1YXRpb24gaGFzIHJ1bi4KICAgICAgICByZWFsX21hcHMgICAgICAgPSBldmFsX2NhbGxiYWNrLm1hcHNbMTpdCiAg'
        'ICAgICAganVzdF9ldmFsdWF0ZWQgID0gbGVuKHJlYWxfbWFwcykgPiAwIGFuZCBldmFsX2NhbGxiYWNrLmVwb2NoZXNbLTFdID09IGVwb2NoICsgMQogICAgICAgIGlmIGp1c3RfZXZhbHVhdGVkIGFuZCBldmFsX2NhbGxiYWNrLm1hcHNbLTFdID49IG1heChyZWFs'
        'X21hcHMpOgogICAgICAgICAgICBwcmludCgnU2F2ZSBiZXN0IG1vZGVsIHRvIGJlc3RfZXBvY2hfd2VpZ2h0cy5wdGggKHZhbCBtQVA9JS4yZiUlKScgJSBldmFsX2NhbGxiYWNrLm1hcHNbLTFdKQogICAgICAgICAgICB0b3JjaC5zYXZlKHNhdmVfc3RhdGVfZGlj'
        'dCwgb3MucGF0aC5qb2luKHNhdmVfZGlyLCAiYmVzdF9lcG9jaF93ZWlnaHRzLnB0aCIpKQogICAgICAgIGVsaWYgbm90IHJlYWxfbWFwcyBhbmQgbG9zcyAvIGVwb2NoX3N0ZXAgPD0gbWluKGxvc3NfaGlzdG9yeS5sb3NzZXMpOgogICAgICAgICAgICBwcmludCgn'
        'U2F2ZSBiZXN0IG1vZGVsIHRvIGJlc3RfZXBvY2hfd2VpZ2h0cy5wdGggKHRyYWluIGxvc3MsIG5vIG1BUCBldmFsIHlldCknKQogICAgICAgICAgICB0b3JjaC5zYXZlKHNhdmVfc3RhdGVfZGljdCwgb3MucGF0aC5qb2luKHNhdmVfZGlyLCAiYmVzdF9lcG9jaF93'
        'ZWlnaHRzLnB0aCIpKQogICAgICAgIHRvcmNoLnNhdmUoc2F2ZV9zdGF0ZV9kaWN0LCBvcy5wYXRoLmpvaW4oc2F2ZV9kaXIsICJsYXN0X2Vwb2NoX3dlaWdodHMucHRoIikpCg=='
    ),
    'eval_one.py': (
        'IiIiClN0YW5kYWxvbmUsIHN1YnByb2Nlc3MtaXNvbGF0ZWQgZXZhbHVhdGlvbiBzY3JpcHQuCgpXaHkgdGhpcyBleGlzdHMgYXMgYSBzZXBhcmF0ZSBzY3JpcHQgaW5zdGVhZCBvZiBhIGZ1bmN0aW9uIGltcG9ydGVkIGludG8gdGhlCm5vdGVib29rJ3MgbG9uZy1s'
        'aXZlZCBrZXJuZWw6IHRoaXMgcHJvamVjdCdzIGFyY2hpdGVjdHVyZSAobmV0cy9tb2RlbC5weSkKcmVhZHMgY29uZmlnLlVTRV9EUk0gYXQgaW1wb3J0IHRpbWUgdG8gZGVjaWRlIHdoZXRoZXIgdGhlIFAzIGJyYW5jaCBpbmNsdWRlcwp0aGUgRGV0YWlsIFJlY292'
        'ZXJ5IE1vZHVsZS4gV2l0aGluIG9uZSBub3RlYm9vayBzZXNzaW9uIHdlIGV2YWx1YXRlCmNoZWNrcG9pbnRzIHRyYWluZWQgdW5kZXIgZGlmZmVyZW50IFVTRV9EUk0gc2V0dGluZ3MgKHRoZSB1bmNoYW5nZWQtYmFzZWxpbmUKY29udHJvbCB2cy4gdGhlIERSTStX'
        'aXNlLUlvVStEV0EgcHJvcG9zZWQgc3lzdGVtKSAtLSBpZiBib3RoIHdlcmUgZXZhbHVhdGVkCnZpYSBgaW1wb3J0IG5ldHMubW9kZWxgIGluIHRoZSBzYW1lIFB5dGhvbiBwcm9jZXNzLCBvbmx5IHRoZSBGSVJTVCBpbXBvcnQKd291bGQgYWN0dWFsbHkgdGFrZSBl'
        'ZmZlY3QgKFB5dGhvbiBjYWNoZXMgbW9kdWxlcyksIHNpbGVudGx5IGV2YWx1YXRpbmcgdGhlCndyb25nIGFyY2hpdGVjdHVyZSBmb3IgdGhlIHNlY29uZCBjaGVja3BvaW50LiBSdW5uaW5nIGVhY2ggZXZhbHVhdGlvbiBhcyBhCmZyZXNoIGBweXRob24gZXZhbF9v'
        'bmUucHkgLi4uYCBzdWJwcm9jZXNzIHNpZGVzdGVwcyB0aGF0IGVudGlyZWx5OiBldmVyeQppbnZvY2F0aW9uIHJlLXJlYWRzIGNvbmZpZy5weSBmcm9tIGRpc2ssIHNvIGl0IGFsd2F5cyBidWlsZHMgdGhlIGV4YWN0CmFyY2hpdGVjdHVyZSB0aGUgY2hlY2twb2lu'
        'dCBiZWluZyBldmFsdWF0ZWQgd2FzIHRyYWluZWQgd2l0aC4KCkJlY2F1c2Ugb2YgdGhpcywgQUxXQVlTIChyZSl3cml0ZSBjb25maWcucHkgd2l0aCB0aGUgY29ycmVjdCBVU0VfRFJNIHZhbHVlCmZvciB0aGUgY2hlY2twb2ludCB5b3UgYXJlIGFib3V0IHRvIGV2'
        'YWx1YXRlIGJlZm9yZSBjYWxsaW5nIHRoaXMgc2NyaXB0LgoKVXNhZ2U6CiAgICBweXRob24gZXZhbF9vbmUucHkgLS1kYXRhbmFtZSBOQU1FIC0tbW9kZWxfcGF0aCBQQVRIIFwKICAgICAgICAtLWltYWdlc19kaXIgRElSIC0tYW5uX2RpciBESVIgLS1pbWFnZV9l'
        'eHQgLmpwZyBcCiAgICAgICAgLS1vdXRfanNvbiByZXN1bHQuanNvbiBcCiAgICAgICAgWy0taWRzX2ZpbGUgcGF0aF93aXRoX29uZV9pZF9wZXJfbGluZV0gXAogICAgICAgIFstLWNsYXNzZXNfcGF0aCBtb2RlbF9kYXRhL3J0dHNfY2xhc3Nlcy50eHRdIFwKICAg'
        'ICAgICBbLS1hbmNob3JzX3BhdGggbW9kZWxfZGF0YS95b2xvX2FuY2hvcnMudHh0XQoKSWYgLS1pZHNfZmlsZSBpcyBvbWl0dGVkLCBldmVyeSBmaWxlIGluIC0taW1hZ2VzX2RpciBtYXRjaGluZyAtLWltYWdlX2V4dCBpcwp1c2VkIGFzIHRoZSBpZCBsaXN0IChp'
        'ZCA9IGZpbGVuYW1lIHdpdGhvdXQgZXh0ZW5zaW9uKS4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCB4bWwuZXRyZWUuRWxlbWVudFRyZWUgYXMgRVQKCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQg'
        'SW1hZ2UsIFVuaWRlbnRpZmllZEltYWdlRXJyb3IKZnJvbSB0cWRtIGltcG9ydCB0cWRtCgpmcm9tIHV0aWxzLnV0aWxzIGltcG9ydCBnZXRfY2xhc3Nlcwpmcm9tIHV0aWxzLnV0aWxzX21hcCBpbXBvcnQgZ2V0X21hcApmcm9tIHlvbG8gaW1wb3J0IFlPTE8KCgpk'
        'ZWYgbGlzdF9pZHNfZnJvbV9kaXIoaW1hZ2VzX2RpciwgZXh0KToKICAgIHJldHVybiBzb3J0ZWQob3MucGF0aC5zcGxpdGV4dChmKVswXSBmb3IgZiBpbiBvcy5saXN0ZGlyKGltYWdlc19kaXIpIGlmIGYubG93ZXIoKS5lbmRzd2l0aChleHQpKQoKCmRlZiBwYXJz'
        'ZV9wZXJfY2xhc3NfYXAocmVzdWx0c190eHRfcGF0aCk6CiAgICAiIiJ1dGlsc19tYXAuZ2V0X21hcCB3cml0ZXMgbGluZXMgbGlrZSAnNjYuMzUlID0gYmljeWNsZSBBUCAnIGludG8KICAgIHJlc3VsdHMudHh0IGluc2lkZSBtYXBfb3V0X3BhdGggLS0gcHVsbCB0'
        'aG9zZSBiYWNrIG91dCBpbnN0ZWFkIG9mCiAgICBtb2RpZnlpbmcgdXRpbHNfbWFwLnB5J3MgYWxyZWFkeS13b3JraW5nIChhbmQgZW50YW5nbGVkKSBpbnRlcm5hbHMuIiIiCiAgICBwZXJfY2xhc3MgPSB7fQogICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHJlc3Vs'
        'dHNfdHh0X3BhdGgpOgogICAgICAgIHJldHVybiBwZXJfY2xhc3MKICAgIHBhdHRlcm4gPSByZS5jb21waWxlKHInXihbXGQuXSspJSA9IChcUyspIEFQXGInKQogICAgd2l0aCBvcGVuKHJlc3VsdHNfdHh0X3BhdGgpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4g'
        'ZjoKICAgICAgICAgICAgbSA9IHBhdHRlcm4ubWF0Y2gobGluZS5zdHJpcCgpKQogICAgICAgICAgICBpZiBtOgogICAgICAgICAgICAgICAgcGVyX2NsYXNzW20uZ3JvdXAoMildID0gZmxvYXQobS5ncm91cCgxKSkKICAgIHJldHVybiBwZXJfY2xhc3MKCgpkZWYg'
        'ZXZhbHVhdGUoZGF0YW5hbWUsIGltYWdlX2lkcywgaW1hZ2VzX2RpciwgYW5ub3RhdGlvbnNfZGlyLCBtb2RlbF9wYXRoLCBpbWFnZV9leHQsCiAgICAgICAgICAgICBjbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aCwgbWluX292ZXJsYXA9MC41LCBjb25maWRlbmNl'
        'PTAuMDAxLAogICAgICAgICAgICAgbm1zX2lvdT0wLjUsIHNjb3JlX3RocmVzaG9sZD0wLjUsIG1hcF9vdXRfcGF0aD1Ob25lKToKICAgIG1hcF9vdXRfcGF0aCA9IG1hcF9vdXRfcGF0aCBvciBmJ21hcF9vdXQte2RhdGFuYW1lfScKICAgIGZvciBzdWIgaW4gWydn'
        'cm91bmQtdHJ1dGgnLCAnZGV0ZWN0aW9uLXJlc3VsdHMnLCAnaW1hZ2VzLW9wdGlvbmFsJ106CiAgICAgICAgb3MubWFrZWRpcnMob3MucGF0aC5qb2luKG1hcF9vdXRfcGF0aCwgc3ViKSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBjbGFzc19uYW1lcywgXyA9IGdldF9j'
        'bGFzc2VzKGNsYXNzZXNfcGF0aCkKICAgIHByaW50KGYiW3tkYXRhbmFtZX1dIHtsZW4oaW1hZ2VfaWRzKX0gaW1hZ2VzIHwgbG9hZGluZyBtb2RlbCBmcm9tIHttb2RlbF9wYXRofSIpCiAgICB5b2xvID0gWU9MTyhtb2RlbF9wYXRoPW1vZGVsX3BhdGgsIGNsYXNz'
        'ZXNfcGF0aD1jbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aD1hbmNob3JzX3BhdGgsCiAgICAgICAgICAgICAgICBjb25maWRlbmNlPWNvbmZpZGVuY2UsIG5tc19pb3U9bm1zX2lvdSwgY3VkYT10b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQoKICAgIHNraXBwZWQg'
        'PSBbXQogICAgZm9yIGltYWdlX2lkIGluIHRxZG0oaW1hZ2VfaWRzLCBkZXNjPWYnW3tkYXRhbmFtZX1dIHByZWRpY3RpbmcnKToKICAgICAgICBpbWFnZV9wYXRoID0gb3MucGF0aC5qb2luKGltYWdlc19kaXIsIGltYWdlX2lkICsgaW1hZ2VfZXh0KQogICAgICAg'
        'IHRyeToKICAgICAgICAgICAgaW1hZ2UgPSBJbWFnZS5vcGVuKGltYWdlX3BhdGgpCiAgICAgICAgICAgIGltYWdlLmxvYWQoKQogICAgICAgIGV4Y2VwdCAoVW5pZGVudGlmaWVkSW1hZ2VFcnJvciwgT1NFcnJvcikgYXMgZToKICAgICAgICAgICAgc2tpcHBlZC5h'
        'cHBlbmQoKGltYWdlX2lkLCBzdHIoZSkpKQogICAgICAgICAgICBvcGVuKG9zLnBhdGguam9pbihtYXBfb3V0X3BhdGgsIGYiZGV0ZWN0aW9uLXJlc3VsdHMve2ltYWdlX2lkfS50eHQiKSwgInciKS5jbG9zZSgpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAg'
        'eW9sby5nZXRfbWFwX3R4dChpbWFnZV9pZCwgaW1hZ2UsIGNsYXNzX25hbWVzLCBtYXBfb3V0X3BhdGgpCgogICAgaWYgc2tpcHBlZDoKICAgICAgICBwcmludChmIlt7ZGF0YW5hbWV9XSBXQVJOSU5HOiBza2lwcGVkIHtsZW4oc2tpcHBlZCl9L3tsZW4oaW1hZ2Vf'
        'aWRzKX0gdW5yZWFkYWJsZSBpbWFnZShzKSAiCiAgICAgICAgICAgICAgZiIodHJlYXRlZCBhcyB6ZXJvIGRldGVjdGlvbnMpIikKICAgICAgICBmb3IgaW1hZ2VfaWQsIGVyciBpbiBza2lwcGVkWzoyMF06CiAgICAgICAgICAgIHByaW50KGYiICAgIHtpbWFnZV9p'
        'ZH17aW1hZ2VfZXh0fToge2Vycn0iKQoKICAgIGJhZF94bWwgPSBbXQogICAgZm9yIGltYWdlX2lkIGluIHRxZG0oaW1hZ2VfaWRzLCBkZXNjPWYnW3tkYXRhbmFtZX1dIGdyb3VuZCB0cnV0aCcpOgogICAgICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4obWFwX291'
        'dF9wYXRoLCBmImdyb3VuZC10cnV0aC97aW1hZ2VfaWR9LnR4dCIpLCAidyIpIGFzIG5ld19mOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByb290ID0gRVQucGFyc2Uob3MucGF0aC5qb2luKGFubm90YXRpb25zX2RpciwgZiJ7aW1hZ2VfaWR9Lnht'
        'bCIpKS5nZXRyb290KCkKICAgICAgICAgICAgZXhjZXB0IEVULlBhcnNlRXJyb3IgYXMgZToKICAgICAgICAgICAgICAgIGJhZF94bWwuYXBwZW5kKChpbWFnZV9pZCwgc3RyKGUpKSkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBvYmog'
        'aW4gcm9vdC5maW5kYWxsKCdvYmplY3QnKToKICAgICAgICAgICAgICAgIGRpZmZpY3VsdF9mbGFnID0gb2JqLmZpbmQoJ2RpZmZpY3VsdCcpIGlzIG5vdCBOb25lIGFuZCBpbnQob2JqLmZpbmQoJ2RpZmZpY3VsdCcpLnRleHQpID09IDEKICAgICAgICAgICAgICAg'
        'IG9ial9uYW1lID0gb2JqLmZpbmQoJ25hbWUnKS50ZXh0CiAgICAgICAgICAgICAgICBpZiBvYmpfbmFtZSBub3QgaW4gY2xhc3NfbmFtZXM6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGJuZCA9IG9iai5maW5kKCdibmRib3gn'
        'KQogICAgICAgICAgICAgICAgbGVmdCwgdG9wID0gYm5kLmZpbmQoJ3htaW4nKS50ZXh0LCBibmQuZmluZCgneW1pbicpLnRleHQKICAgICAgICAgICAgICAgIHJpZ2h0LCBib3R0b20gPSBibmQuZmluZCgneG1heCcpLnRleHQsIGJuZC5maW5kKCd5bWF4JykudGV4'
        'dAogICAgICAgICAgICAgICAgc3VmZml4ID0gIiBkaWZmaWN1bHQiIGlmIGRpZmZpY3VsdF9mbGFnIGVsc2UgIiIKICAgICAgICAgICAgICAgIG5ld19mLndyaXRlKGYie29ial9uYW1lfSB7bGVmdH0ge3RvcH0ge3JpZ2h0fSB7Ym90dG9tfXtzdWZmaXh9XG4iKQoK'
        'ICAgIGlmIGJhZF94bWw6CiAgICAgICAgcHJpbnQoZiJbe2RhdGFuYW1lfV0gV0FSTklORzoge2xlbihiYWRfeG1sKX0gdW5wYXJzZWFibGUgYW5ub3RhdGlvbihzKSAodHJlYXRlZCBhcyAwIG9iamVjdHMpIikKCiAgICBwcmludChmIlt7ZGF0YW5hbWV9XSBjb21w'
        'dXRpbmcgbUFQIikKICAgICMgTk9URTogZ2V0X21hcCByZXR1cm5zIG1BUCBhcyBhIDAtMSBmcmFjdGlvbiAoZS5nLiAwLjc4MzkpLCBub3QgYQogICAgIyBwZXJjZW50YWdlIC0tIGRlc3BpdGUgdGhlIGV4aXN0aW5nIG5vdGVib29rcycgZXZhbHVhdGUoKSBwcmlu'
        'dGluZyBpdAogICAgIyB3aXRoIGEgdHJhaWxpbmcgJyUnIHVuc2NhbGVkIChhIHByZS1leGlzdGluZyBmb3JtYXR0aW5nIGluY29uc2lzdGVuY3kKICAgICMgaW4gdGhpcyByZXBvLCBub3QgaW50cm9kdWNlZCBoZXJlKS4gV2UgbXVsdGlwbHkgYnkgMTAwIGV4cGxp'
        'Y2l0bHkgYmVsb3cKICAgICMgc28gcmVzdWx0cyBpbiBvdXRfanNvbiBhcmUgdW5hbWJpZ3VvdXNseSBwZXJjZW50YWdlcy4KICAgIG1BUF9mcmFjdGlvbiA9IGdldF9tYXAobWluX292ZXJsYXAsIFRydWUsIHNjb3JlX3RocmVob2xkPXNjb3JlX3RocmVzaG9sZCwg'
        'cGF0aD1tYXBfb3V0X3BhdGgpCiAgICBwZXJfY2xhc3MgPSBwYXJzZV9wZXJfY2xhc3NfYXAob3MucGF0aC5qb2luKG1hcF9vdXRfcGF0aCwgJ3Jlc3VsdHMudHh0JykpCiAgICByZXR1cm4gbUFQX2ZyYWN0aW9uICogMTAwLjAsIHBlcl9jbGFzcwoKCmRlZiBtYWlu'
        'KCk6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcC5hZGRfYXJndW1lbnQoJy0tZGF0YW5hbWUnLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoJy0tbW9kZWxfcGF0aCcsIHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1'
        'bWVudCgnLS1pbWFnZXNfZGlyJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCctLWFubl9kaXInLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoJy0taW1hZ2VfZXh0JywgZGVmYXVsdD0nLmpwZycpCiAgICBwLmFkZF9hcmd1bWVu'
        'dCgnLS1pZHNfZmlsZScsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSdvcHRpb25hbCBmaWxlIHdpdGggb25lIGltYWdlIGlkIHBlciBsaW5lOyBkZWZhdWx0cyB0byBsaXN0aW5nIGltYWdlc19kaXInKQogICAgcC5hZGRfYXJndW1lbnQo'
        'Jy0tY2xhc3Nlc19wYXRoJywgZGVmYXVsdD0nbW9kZWxfZGF0YS9ydHRzX2NsYXNzZXMudHh0JykKICAgIHAuYWRkX2FyZ3VtZW50KCctLWFuY2hvcnNfcGF0aCcsIGRlZmF1bHQ9J21vZGVsX2RhdGEveW9sb19hbmNob3JzLnR4dCcpCiAgICBwLmFkZF9hcmd1bWVu'
        'dCgnLS1taW5fb3ZlcmxhcCcsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC41KQogICAgcC5hZGRfYXJndW1lbnQoJy0tY29uZmlkZW5jZScsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wMDEpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1ubXNfaW91JywgdHlwZT1mbG9hdCwg'
        'ZGVmYXVsdD0wLjUpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1zY29yZV90aHJlc2hvbGQnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNSkKICAgIHAuYWRkX2FyZ3VtZW50KCctLW91dF9qc29uJywgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwLnBhcnNlX2FyZ3Mo'
        'KQoKICAgIGlmIGFyZ3MuaWRzX2ZpbGU6CiAgICAgICAgIyBNYXRjaGVzIHJkZm5ldF9iYXNlbGluZV9ldmFsLmlweW5iJ3MgcmVhZF90ZXN0X2lkcygpIGNvbnZlbnRpb24gZXhhY3RseQogICAgICAgICMgKHdoaXRlc3BhY2Utc3BsaXQsIG5vdCBsaW5lLXNwbGl0'
        'KSBmb3IgY29uc2lzdGVuY3kgd2l0aCB0aGUgYWxyZWFkeS0KICAgICAgICAjIHZhbGlkYXRlZCBiYXNlbGluZSBldmFsdWF0aW9uLgogICAgICAgIGltYWdlX2lkcyA9IG9wZW4oYXJncy5pZHNfZmlsZSkucmVhZCgpLnN0cmlwKCkuc3BsaXQoKQogICAgZWxzZToK'
        'ICAgICAgICBpbWFnZV9pZHMgPSBsaXN0X2lkc19mcm9tX2RpcihhcmdzLmltYWdlc19kaXIsIGFyZ3MuaW1hZ2VfZXh0KQoKICAgIG1BUCwgcGVyX2NsYXNzID0gZXZhbHVhdGUoCiAgICAgICAgZGF0YW5hbWU9YXJncy5kYXRhbmFtZSwgaW1hZ2VfaWRzPWltYWdl'
        'X2lkcywgaW1hZ2VzX2Rpcj1hcmdzLmltYWdlc19kaXIsCiAgICAgICAgYW5ub3RhdGlvbnNfZGlyPWFyZ3MuYW5uX2RpciwgbW9kZWxfcGF0aD1hcmdzLm1vZGVsX3BhdGgsIGltYWdlX2V4dD1hcmdzLmltYWdlX2V4dCwKICAgICAgICBjbGFzc2VzX3BhdGg9YXJn'
        'cy5jbGFzc2VzX3BhdGgsIGFuY2hvcnNfcGF0aD1hcmdzLmFuY2hvcnNfcGF0aCwKICAgICAgICBtaW5fb3ZlcmxhcD1hcmdzLm1pbl9vdmVybGFwLCBjb25maWRlbmNlPWFyZ3MuY29uZmlkZW5jZSwKICAgICAgICBubXNfaW91PWFyZ3Mubm1zX2lvdSwgc2NvcmVf'
        'dGhyZXNob2xkPWFyZ3Muc2NvcmVfdGhyZXNob2xkLAogICAgICAgIG1hcF9vdXRfcGF0aD1mJy9rYWdnbGUvd29ya2luZy9tYXBfb3V0LXthcmdzLmRhdGFuYW1lfScsCiAgICApCgogICAgcmVzdWx0ID0geydkYXRhbmFtZSc6IGFyZ3MuZGF0YW5hbWUsICdtb2Rl'
        'bF9wYXRoJzogYXJncy5tb2RlbF9wYXRoLAogICAgICAgICAgICAgICdudW1faW1hZ2VzJzogbGVuKGltYWdlX2lkcyksICdtQVAnOiBtQVAsICdwZXJfY2xhc3NfQVAnOiBwZXJfY2xhc3N9CiAgICB3aXRoIG9wZW4oYXJncy5vdXRfanNvbiwgJ3cnKSBhcyBmOgog'
        'ICAgICAgIGpzb24uZHVtcChyZXN1bHQsIGYsIGluZGVudD0yKQogICAgcHJpbnQoZiJcbj09PSBbe2FyZ3MuZGF0YW5hbWV9XSBtQVBAe2FyZ3MubWluX292ZXJsYXB9OiB7bUFQOi4yZn0lID09PSIpCiAgICBmb3IgY2xzLCBhcCBpbiBzb3J0ZWQocGVyX2NsYXNz'
        'Lml0ZW1zKCkpOgogICAgICAgIHByaW50KGYiICAgIHtjbHM6MTJzfToge2FwOi4yZn0lIikKICAgIHByaW50KGYiUmVzdWx0IHdyaXR0ZW4gdG8ge2FyZ3Mub3V0X2pzb259IikKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgbWFpbigpCg=='
    ),
}

for _rel, _b64 in _FILES_B64.items():
    _path = _rel
    os.makedirs(os.path.dirname(_path), exist_ok=True) if os.path.dirname(_path) else None
    with open(_path, 'wb') as _f:
        _f.write(base64.b64decode(_b64))
    print('Wrote', _path, '(', len(base64.b64decode(_b64)), 'bytes )')

print('\nAll thesis modification files written -- no git push of the code changes was required.')

In [ ]:
!pip install -q opencv-python-headless tqdm thop
import torch, json, os, sys, glob, time, subprocess

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
n_gpus = torch.cuda.device_count()
print(f"GPU count: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpus < 2:
    print("\nWARNING: this notebook launches training with torchrun --nproc_per_node=2 (DDP)\n"
          "and nets/backbone.py's forward hardcodes an 8+8 split that assumes exactly 2 GPUs.\n"
          "Go to Settings -> Accelerator -> GPU T4 x2 and restart the kernel.")

## Resolve dataset paths

Same resolution logic as `rdfnet_train_eval.ipynb` (VOC-FOG) and `rdfnet_baseline_eval.ipynb`
(RTTS, RDFNet.pth) -- searches for the directory that actually contains the expected
subfolders rather than assuming a fixed nesting depth, since these dataset uploads have
shifted layout before.

In [ ]:
def find_data_root(base, required_subdirs):
    for dirpath, dirnames, _ in os.walk(base):
        if all(d in dirnames for d in required_subdirs):
            return dirpath
    raise FileNotFoundError(f"No directory under {base} contains all of {required_subdirs}")

def find_weight_file(base):
    if os.path.isfile(base) and base.endswith('.pth'):
        return base
    matches = glob.glob(os.path.join(base, '**', '*.pth'), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No .pth weight file found under {base}.")
    return matches[0]

def read_test_ids(test_txt_path):
    return open(test_txt_path).read().strip().split()

# --- VOC-FOG: separate train/ and test/ folders ---
VOC_FOG_BASE = "/kaggle/input/datasets/mdhabibourrahman/voc-fog/VOC_FOG"
TRAIN_ROOT = find_data_root(VOC_FOG_BASE, ("Annotations", "JPEGImages", "VOC2007-FOG"))
TRAIN_FOG_DIR   = os.path.join(TRAIN_ROOT, "VOC2007-FOG")
TRAIN_CLEAN_DIR = os.path.join(TRAIN_ROOT, "JPEGImages")
TRAIN_ANN_DIR   = os.path.join(TRAIN_ROOT, "Annotations")
TEST_ROOT = find_data_root(VOC_FOG_BASE, ("Annotations", "VOCtest-FOG"))
TEST_FOG_DIR = os.path.join(TEST_ROOT, "VOCtest-FOG")
TEST_ANN_DIR = os.path.join(TEST_ROOT, "Annotations")
print(f"VOC-FOG train: fog={TRAIN_FOG_DIR}\n               clean={TRAIN_CLEAN_DIR}\n               ann={TRAIN_ANN_DIR}")
print(f"VOC-FOG test : fog={TEST_FOG_DIR}\n               ann={TEST_ANN_DIR}")

# --- RTTS: flat layout, split defined by ImageSets/Main/test.txt, images are .png ---
RTTS_BASE       = "/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS"
RTTS_IMAGES_DIR = os.path.join(RTTS_BASE, "JPEGImages")
RTTS_ANN_DIR    = os.path.join(RTTS_BASE, "Annotations")
RTTS_IDS        = read_test_ids(os.path.join(RTTS_BASE, "ImageSets", "Main", "test.txt"))
print(f"RTTS: {len(RTTS_IDS)} images | images_dir={RTTS_IMAGES_DIR}")

# --- Base paper's converged checkpoint (fine-tune starting point) ---
BASELINE_PTH = find_weight_file("/kaggle/input/datasets/mdhabibourrahman/rdfnet-baseline")
print(f"BASELINE_PTH: {BASELINE_PTH}")

## Build VOC-FOG train/val annotation files

`train.py` reads `train_annotation_path` (foggy) and `clear_annotation_path` (clean), which
must be index-aligned pairs of the same underlying photo, plus `val_annotation_path` (foggy
test images, used both by the periodic `EvalCallback` during training and reused below as the
final VOC-FOG-test set). Identical logic to `rdfnet_train_eval.ipynb`.

In [ ]:
import re
import xml.etree.ElementTree as ET
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

CLASSES_PATH = 'model_data/rtts_classes.txt'
with open(CLASSES_PATH) as f:
    CLASSES = [c.strip() for c in f if c.strip()]
print(f"Classes ({len(CLASSES)}): {CLASSES}")

FOG_SUFFIX_RE = re.compile(r'^(?P<base>.+?)_f\d+$')

def base_stem(stem):
    m = FOG_SUFFIX_RE.match(stem)
    return m.group('base') if m else stem

def is_valid_image(path):
    try:
        with Image.open(path) as im:
            im.load()
        return True
    except (UnidentifiedImageError, OSError):
        return False

def parse_boxes(xml_path):
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return None
    boxes = []
    for obj in root.iter('object'):
        diff = obj.find('difficult')
        if diff is not None and int(diff.text) == 1:
            continue
        cls = obj.find('name').text.strip()
        if cls not in CLASSES:
            continue
        bb = obj.find('bndbox')
        coords = [int(float(bb.find(t).text)) for t in ('xmin', 'ymin', 'xmax', 'ymax')]
        boxes.append(','.join(map(str, coords)) + ',' + str(CLASSES.index(cls)))
    return boxes

def box_suffix(boxes):
    return ''.join(' ' + b for b in boxes)

def build_train_pairs(fog_dir, clean_dir, ann_dir, out_fog_path, out_clean_path):
    fog_lines, clean_lines = [], []
    skipped = {'no_clean': 0, 'no_ann': 0, 'no_boxes': 0, 'bad_fog_img': 0, 'bad_clean_img': 0}
    fog_files = sorted(f for f in os.listdir(fog_dir) if f.lower().endswith('.jpg'))
    for fname in tqdm(fog_files, desc='pairing train fog/clean'):
        fog_stem = os.path.splitext(fname)[0]
        stem = base_stem(fog_stem)
        clean_path = os.path.join(clean_dir, stem + '.jpg')
        ann_path = os.path.join(ann_dir, stem + '.xml')
        fog_path = os.path.join(fog_dir, fname)
        if not os.path.exists(clean_path):
            skipped['no_clean'] += 1
            continue
        if not os.path.exists(ann_path):
            skipped['no_ann'] += 1
            continue
        boxes = parse_boxes(ann_path)
        if not boxes:
            skipped['no_boxes'] += 1
            continue
        if not is_valid_image(fog_path):
            skipped['bad_fog_img'] += 1
            continue
        if not is_valid_image(clean_path):
            skipped['bad_clean_img'] += 1
            continue
        suffix = box_suffix(boxes)
        fog_lines.append(fog_path + suffix)
        clean_lines.append(clean_path + suffix)
    with open(out_fog_path, 'w') as f:
        f.write('\n'.join(fog_lines) + '\n')
    with open(out_clean_path, 'w') as f:
        f.write('\n'.join(clean_lines) + '\n')
    print(f"train: {len(fog_lines)} pairs written -> {out_fog_path}, {out_clean_path} | skipped: {skipped}")
    return len(fog_lines)

def build_val(fog_dir, ann_dir, out_path):
    lines, ids = [], []
    skipped = {'no_ann': 0, 'no_boxes': 0, 'bad_img': 0}
    fog_files = sorted(f for f in os.listdir(fog_dir) if f.lower().endswith('.jpg'))
    for fname in tqdm(fog_files, desc='building val/test list'):
        fog_stem = os.path.splitext(fname)[0]
        stem = base_stem(fog_stem)
        ann_path = os.path.join(ann_dir, stem + '.xml')
        fog_path = os.path.join(fog_dir, fname)
        if not os.path.exists(ann_path):
            skipped['no_ann'] += 1
            continue
        boxes = parse_boxes(ann_path)
        if not boxes:
            skipped['no_boxes'] += 1
            continue
        if not is_valid_image(fog_path):
            skipped['bad_img'] += 1
            continue
        lines.append(fog_path + box_suffix(boxes))
        ids.append(fog_stem)
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f"val/test: {len(lines)} images written -> {out_path} | skipped: {skipped}")
    return ids

NUM_TRAIN_PAIRS = build_train_pairs(TRAIN_FOG_DIR, TRAIN_CLEAN_DIR, TRAIN_ANN_DIR,
                                     '2007_train_fog.txt', '2007_train.txt')
VOCFOG_TEST_IDS = build_val(TEST_FOG_DIR, TEST_ANN_DIR, '2007_val_fog.txt')
print(f"\nTrain pairs: {NUM_TRAIN_PAIRS} | VOC-FOG test images: {len(VOCFOG_TEST_IDS)}")

## Config writer and fine-tune epoch budget

`write_config(...)` writes `config.py` from scratch for a given run -- this is how the notebook
switches between the baseline / control / proposed architectures and losses, since
`nets/model.py`, `nets/yolo_training.py`, and `utils/utils_fit.py` all read their
`USE_DRM` / `USE_WISE_IOU` / `USE_DWA` toggles from `config.py` at import/construction time.

**`FINETUNE_EPOCHS` is a placeholder (60) pending your own timing calibration.** Run the
control training cell below, watch the first epoch's `tqdm` bar for `s/it` or `it/s`, multiply
by ~510 steps/epoch (8172 train pairs / batch 16), and that's your real per-epoch time -- if
your total budget doesn't comfortably fit two runs of `FINETUNE_EPOCHS` epochs each at that
rate, lower this number and restart from this cell (both runs must use the *same*
`FINETUNE_EPOCHS` for the control comparison to remain valid -- see the notebook's markdown
intro).

In [ ]:
FINETUNE_EPOCHS = 60  # <-- adjust after timing the first epoch; keep IDENTICAL for both runs

def write_config(save_dir, use_drm, use_wiou, use_dwa, model_path=None,
                  freeze_train=False, init_lr=1e-3, unfreeze_epochs=FINETUNE_EPOCHS,
                  eval_period=5, num_workers=2, distributed=True):
    model_path = model_path or BASELINE_PTH
    config_content = f'''import os

Cuda            = True
seed            = 114514
distributed     = {distributed}
sync_bn         = False
fp16            = False

classes_path    = 'model_data/rtts_classes.txt'
anchors_path    = 'model_data/yolo_anchors.txt'
anchors_mask    = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
model_path      = "{model_path}"
input_shape     = [640, 640]

pretrained      = False
Init_Epoch          = 0
Freeze_Epoch        = 100
Freeze_batch_size   = 16
UnFreeze_Epoch      = {unfreeze_epochs}
Unfreeze_batch_size = 16
Freeze_Train        = {freeze_train}
Init_lr             = {init_lr}
Min_lr              = Init_lr * 0.01
optimizer_type      = "sgd"
momentum            = 0.937
weight_decay        = 5e-4
lr_decay_type       = "cos"
save_period         = 10
save_dir            = "{save_dir}"
eval_flag           = True
eval_period         = {eval_period}
num_workers         = {num_workers}

train_annotation_path = "2007_train_fog.txt"
val_annotation_path   = "2007_val_fog.txt"
clear_annotation_path = "2007_train.txt"

USE_DRM         = {use_drm}
USE_WISE_IOU    = {use_wiou}
USE_DWA         = {use_dwa}
'''
    with open('config.py', 'w') as f:
        f.write(config_content)
    for k in list(sys.modules.keys()):
        if k == 'config':
            del sys.modules[k]
    import config as cfg
    print(f"config.py written | save_dir={cfg.save_dir} model_path={cfg.model_path}")
    print(f"  USE_DRM={cfg.USE_DRM} USE_WISE_IOU={cfg.USE_WISE_IOU} USE_DWA={cfg.USE_DWA}")
    print(f"  Freeze_Train={cfg.Freeze_Train} Init_lr={cfg.Init_lr} UnFreeze_Epoch={cfg.UnFreeze_Epoch}")
    print(f"  eval_period={cfg.eval_period} num_workers={cfg.num_workers}")
    return cfg

def run_eval(dataname, model_path, images_dir, ann_dir, image_ext, use_drm, ids_file=None):
    """Rewrites config.py's USE_DRM to match the checkpoint being evaluated, then
    runs eval_one.py as a FRESH subprocess (not an in-kernel import) so the
    architecture nets/model.py builds always matches what this checkpoint was
    trained with -- see eval_one.py's docstring."""
    write_config(save_dir='logs_eval_scratch', use_drm=use_drm, use_wiou=False, use_dwa=False,
                 model_path=model_path)
    out_json = f'/kaggle/working/eval_{dataname}_{os.path.basename(model_path)}.json'
    cmd = ['python', 'eval_one.py', '--dataname', dataname, '--model_path', model_path,
           '--images_dir', images_dir, '--ann_dir', ann_dir, '--image_ext', image_ext,
           '--out_json', out_json]
    if ids_file:
        cmd += ['--ids_file', ids_file]
    print('Running:', ' '.join(cmd))
    ret = subprocess.run(cmd)
    if ret.returncode != 0:
        raise RuntimeError(f"eval_one.py failed for {dataname} / {model_path}")
    with open(out_json) as f:
        return json.load(f)

## Baseline reference (no fine-tuning)

Evaluates the base paper's own `RDFNet.pth` checkpoint as-is, using the same
`eval_one.py` path as the control/proposed checkpoints below, so all three numbers in the
final table come from an identical evaluation pipeline.

In [ ]:
RTTS_TEST_TXT = os.path.join(RTTS_BASE, "ImageSets", "Main", "test.txt")

baseline_vocfog = run_eval('VOC-FOG-test', BASELINE_PTH, TEST_FOG_DIR, TEST_ANN_DIR, '.jpg', use_drm=False)
baseline_rtts   = run_eval('RTTS', BASELINE_PTH, RTTS_IMAGES_DIR, RTTS_ANN_DIR, '.png', use_drm=False,
                            ids_file=RTTS_TEST_TXT)
print(f"\nBaseline  | VOC-FOG mAP={baseline_vocfog['mAP']:.2f}%  RTTS mAP={baseline_rtts['mAP']:.2f}%")

## Control run: unchanged architecture and loss, continued training

Same starting checkpoint, same epoch budget as the proposed run below -- this isolates
"does training longer alone help?" from "does the technique help?" (see notebook intro).

In [ ]:
write_config(save_dir='logs_control', use_drm=False, use_wiou=False, use_dwa=False,
             model_path=BASELINE_PTH, freeze_train=False, init_lr=1e-3)

t_start = time.time()
ret = os.system(f'torchrun --nproc_per_node={max(torch.cuda.device_count(), 1)} --master_port=12355 train.py')
elapsed = (time.time() - t_start) / 3600
print(f"\n[control] training exit code: {ret} | elapsed: {elapsed:.2f}h")
if ret != 0:
    print("Training did not exit cleanly -- check the log above before proceeding.")

In [ ]:
control_weights = sorted(glob.glob('logs_control/best_epoch_weights.pth')) or sorted(glob.glob('logs_control/last_epoch_weights.pth'))
if not control_weights:
    raise FileNotFoundError("No checkpoint found under logs_control/ -- did training complete at least one epoch?")
CONTROL_PTH = control_weights[-1]
print('Using control weights:', CONTROL_PTH)

control_vocfog = run_eval('VOC-FOG-test', CONTROL_PTH, TEST_FOG_DIR, TEST_ANN_DIR, '.jpg', use_drm=False)
control_rtts   = run_eval('RTTS', CONTROL_PTH, RTTS_IMAGES_DIR, RTTS_ANN_DIR, '.png', use_drm=False,
                           ids_file=RTTS_TEST_TXT)
print(f"\nControl   | VOC-FOG mAP={control_vocfog['mAP']:.2f}%  RTTS mAP={control_rtts['mAP']:.2f}%")

## Proposed run: DRM + Wise-IoU + DWA, continued training

Identical starting checkpoint and epoch budget to the control run above -- only the
architecture (`USE_DRM`) and loss formulation (`USE_WISE_IOU`, `USE_DWA`) differ.

In [ ]:
write_config(save_dir='logs_proposed', use_drm=True, use_wiou=True, use_dwa=True,
             model_path=BASELINE_PTH, freeze_train=False, init_lr=1e-3)

t_start = time.time()
ret = os.system(f'torchrun --nproc_per_node={max(torch.cuda.device_count(), 1)} --master_port=12355 train.py')
elapsed = (time.time() - t_start) / 3600
print(f"\n[proposed] training exit code: {ret} | elapsed: {elapsed:.2f}h")
if ret != 0:
    print("Training did not exit cleanly -- check the log above before proceeding.")

In [ ]:
proposed_weights = sorted(glob.glob('logs_proposed/best_epoch_weights.pth')) or sorted(glob.glob('logs_proposed/last_epoch_weights.pth'))
if not proposed_weights:
    raise FileNotFoundError("No checkpoint found under logs_proposed/ -- did training complete at least one epoch?")
PROPOSED_PTH = proposed_weights[-1]
print('Using proposed weights:', PROPOSED_PTH)

proposed_vocfog = run_eval('VOC-FOG-test', PROPOSED_PTH, TEST_FOG_DIR, TEST_ANN_DIR, '.jpg', use_drm=True)
proposed_rtts   = run_eval('RTTS', PROPOSED_PTH, RTTS_IMAGES_DIR, RTTS_ANN_DIR, '.png', use_drm=True,
                            ids_file=RTTS_TEST_TXT)
print(f"\nProposed  | VOC-FOG mAP={proposed_vocfog['mAP']:.2f}%  RTTS mAP={proposed_rtts['mAP']:.2f}%")

## Final comparison table

In [ ]:
rows = [
    ('Baseline (paper, 0 extra epochs)', baseline_vocfog, baseline_rtts),
    (f'Control (+{FINETUNE_EPOCHS} epochs, unchanged)', control_vocfog, control_rtts),
    (f'Proposed (+{FINETUNE_EPOCHS} epochs, DRM+WIoU+DWA)', proposed_vocfog, proposed_rtts),
]

print(f"{'Method':42s} {'VOC-FOG mAP':>12s} {'RTTS mAP':>10s}")
for name, vf, rt in rows:
    print(f"{name:42s} {vf['mAP']:11.2f}% {rt['mAP']:9.2f}%")

print("\nPer-class AP on RTTS (bicycle/motorbike are DRM's target classes):")
all_classes = sorted(set(baseline_rtts['per_class_AP']) | set(control_rtts['per_class_AP']) | set(proposed_rtts['per_class_AP']))
print(f"{'class':12s} {'baseline':>10s} {'control':>10s} {'proposed':>10s}")
for c in all_classes:
    b = baseline_rtts['per_class_AP'].get(c, float('nan'))
    ct = control_rtts['per_class_AP'].get(c, float('nan'))
    p = proposed_rtts['per_class_AP'].get(c, float('nan'))
    print(f"{c:12s} {b:9.2f}% {ct:9.2f}% {p:9.2f}%")

print("\nPer-class AP on VOC-FOG-test:")
all_classes_vf = sorted(set(baseline_vocfog['per_class_AP']) | set(control_vocfog['per_class_AP']) | set(proposed_vocfog['per_class_AP']))
print(f"{'class':12s} {'baseline':>10s} {'control':>10s} {'proposed':>10s}")
for c in all_classes_vf:
    b = baseline_vocfog['per_class_AP'].get(c, float('nan'))
    ct = control_vocfog['per_class_AP'].get(c, float('nan'))
    p = proposed_vocfog['per_class_AP'].get(c, float('nan'))
    print(f"{c:12s} {b:9.2f}% {ct:9.2f}% {p:9.2f}%")

## Model complexity (Params/FLOPs) for the control vs. proposed architecture

Confirms how small DRM's added footprint actually is -- everything else (Wise-IoU, DWA) adds
zero parameters since they only change the loss computation, not the network.

In [ ]:
from nets.model import YoloBody
from utils.utils import get_anchors, get_classes
from thop import profile

class_names, num_classes = get_classes('model_data/rtts_classes.txt')
anchors, num_anchors = get_anchors('model_data/yolo_anchors.txt')
anchors_mask = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
dummy = torch.randn(1, 3, 640, 640)

for label, use_drm in [('control (USE_DRM=False)', False), ('proposed (USE_DRM=True)', True)]:
    write_config(save_dir='logs_eval_scratch', use_drm=use_drm, use_wiou=False, use_dwa=False,
                 model_path=BASELINE_PTH)
    for k in list(sys.modules.keys()):
        if k.startswith('nets.model') or k == 'config':
            del sys.modules[k]
    from nets.model import YoloBody as _YoloBody
    m = _YoloBody(anchors_mask, num_classes)
    m.eval()
    flops, params = profile(m, inputs=(dummy,), verbose=False)
    print(f"{label:28s} Params: {params/1e6:.3f}M  FLOPs: {flops/1e9:.3f}G")

## Save outputs

In [ ]:
import shutil

out_dir = '/kaggle/working/rdfnet_finetune_results'
os.makedirs(out_dir, exist_ok=True)

for pattern in ['logs_control/best_epoch_weights.pth', 'logs_control/last_epoch_weights.pth',
                'logs_control/epoch_map.txt', 'logs_control/epoch_map.png', 'logs_control/epoch_loss.png',
                'logs_proposed/best_epoch_weights.pth', 'logs_proposed/last_epoch_weights.pth',
                'logs_proposed/epoch_map.txt', 'logs_proposed/epoch_map.png', 'logs_proposed/epoch_loss.png',
                'eval_*.json']:
    for f in glob.glob(pattern):
        dest = os.path.join(out_dir, f.replace('/', '_'))
        shutil.copy(f, dest)
        print('Saved:', dest)

summary = {
    'finetune_epochs': FINETUNE_EPOCHS,
    'baseline': {'VOC-FOG': baseline_vocfog, 'RTTS': baseline_rtts},
    'control':  {'VOC-FOG': control_vocfog,  'RTTS': control_rtts},
    'proposed': {'VOC-FOG': proposed_vocfog, 'RTTS': proposed_rtts},
}
with open(os.path.join(out_dir, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nDone. Download from Kaggle Output tab -> {os.path.basename(out_dir)}/")